# MoMoK Ablation
Dataset: `MKG-W`
Variant: `no_gating_noise`
Pinned source: `99c2df114d48c79708ea0608644b685183d81bd9`


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
REPO_URL='https://github.com/zjukg/MoMoK.git'
SOURCE_COMMIT='99c2df114d48c79708ea0608644b685183d81bd9'
DATASET='MKG-W'
VARIANT='no_gating_noise'
EXPECTED={'entities': 15000, 'relations': 169, 'train': 34196, 'valid': 4276, 'test': 4274}
OUTPUT_DIR=Path('/kaggle/working/momok_ablation')
REPO_DIR=Path('/kaggle/working/MoMoK')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
subprocess.check_call(['git','clone',REPO_URL,str(REPO_DIR)])
subprocess.check_call(['git','-C',str(REPO_DIR),'checkout','--detach',SOURCE_COMMIT])
os.environ['MOMOK_SOURCE_COMMIT']=SOURCE_COMMIT
print('SOURCE_PIN=PASS')


In [ ]:
OVERLAY={'train.py': 'import argparse\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom tqdm import tqdm\n\nfrom ablations.runtime_contract import estimator_should_run\nfrom ablations.torch_checkpoint import build_payload, checkpoint_config, restore_checkpoint, save_checkpoint\nfrom ablations.variants import resolve_variant\nfrom models.RelMoE import RelMoE\nfrom models.model import *\nfrom models.modules import MIEstimator\nfrom utils.data_loader import *\nfrom utils.data_util import load_data\n\n\ndef parse_args():\n    config_args = {\n        \'lr\': 0.0005,\n        \'dropout_gat\': 0.3,\n        \'dropout\': 0.3,\n        \'cuda\': 0,\n        \'epochs_gat\': 3000,\n        \'epochs\': 2000,\n        \'weight_decay_gat\': 1e-5,\n        \'weight_decay\': 0,\n        \'seed\': 10010,\n        \'model\': \'RMoE\',\n        \'num-layers\': 3,\n        \'dim\': 256,\n        \'r_dim\': 256,\n        \'k_w\': 10,\n        \'k_h\': 20,\n        \'n_heads\': 2,\n        \'dataset\': \'DB15K\',\n        \'pre_trained\': 0,\n        \'encoder\': 0,\n        \'image_features\': 1,\n        \'text_features\': 1,\n        \'patience\': 5,\n        \'eval_freq\': 100,\n        \'lr_reduce_freq\': 500,\n        \'gamma\': 1.0,\n        \'bias\': 1,\n        \'neg_num\': 2,\n        \'neg_num_gat\': 2,\n        \'alpha\': 0.2,\n        \'alpha_gat\': 0.2,\n        \'out_channels\': 32,\n        \'kernel_size\': 3,\n        \'batch_size\': 1024,\n        \'save\': 1,\n        \'n_exp\': 3,\n        \'mu\': 0.0001,\n        \'img_dim\': 256,\n        \'txt_dim\': 256,\n        \'variant\': \'full\',\n        \'max_train_batches\': 0,\n        \'resume_checkpoint\': \'\',\n        \'output_dir\': \'\',\n        \'run_mode\': \'full\',\n        \'metric_policy\': \'legacy\',\n        \'wandb_mode\': \'disabled\'\n        , \'checkpoint_dir\': \'\'\n        , \'wandb_run_id\': \'\'\n    }\n\n    parser = argparse.ArgumentParser()\n    for param, val in config_args.items():\n        parser.add_argument(f"--{param}", default=val, type=type(val))\n    args = parser.parse_args()\n    return args\n\nargs = parse_args()\nargs.variant_config = resolve_variant(args.variant, n_exp=args.n_exp, mu=args.mu)\nprint(args)\nnp.random.seed(args.seed)\ntorch.manual_seed(args.seed)\nargs.device = \'cuda:\' + str(args.cuda) if int(args.cuda) >= 0 else \'cpu\'\nprint(f\'Using: {args.device}\')\ntorch.cuda.set_device(args.cuda)\nfor k, v in list(vars(args).items()):\n    print(str(k) + \':\' + str(v))\n\nentity2id, relation2id, img_features, text_features, train_data, val_data, test_data = load_data(args.dataset)\nprint("Training data {:04d}".format(len(train_data[0])))\n\ncorpus = ConvECorpus(args, train_data, val_data, test_data, entity2id, relation2id)\n\nif args.image_features:\n    args.img = F.normalize(torch.Tensor(img_features), p=2, dim=1)\nif args.text_features:\n    args.desp = F.normalize(torch.Tensor(text_features), p=2, dim=1)\nargs.entity2id = entity2id\nargs.relation2id = relation2id\n\nmodel_name = {\n    \'RMoE\': RelMoE\n}\ntime.sleep(5)\n\n\n\ndef train_decoder(args):\n    model = model_name[args.model](args)\n    args.img_dim = model.img_dim\n    args.txt_dim = model.txt_dim\n    estimator = MIEstimator(args) if estimator_should_run(args.variant_config) else None\n    print(str(model))\n    optimizer = torch.optim.Adam(params=model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, args.gamma)\n    optimizer_mi = (\n        torch.optim.Adam(params=estimator.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n        if estimator is not None\n        else None\n    )\n    tot_params = sum([np.prod(p.size()) for p in model.parameters()])\n    print(f\'Total number of parameters: {tot_params}\')\n    if args.cuda is not None and int(args.cuda) >= 0:\n        model = model.to(args.device)\n        if estimator is not None:\n            estimator = estimator.to(args.device)\n\n    # Train Model\n    t_total = time.time()\n    counter = 0\n    best_val_metrics = model.init_metric_dict()\n    best_test_metrics = model.init_metric_dict()\n    start_epoch = 0\n    if args.resume_checkpoint:\n        payload = restore_checkpoint(\n            args.resume_checkpoint,\n            args=args,\n            model=model,\n            estimator=estimator,\n            optimizer=optimizer,\n            optimizer_mi=optimizer_mi,\n            lr_scheduler=lr_scheduler,\n            corpus=corpus,\n        )\n        best_test_metrics = payload["best_validation_metrics"]\n        start_epoch = int(payload["completed_epoch"])\n        print(f"RESUME_OK={args.resume_checkpoint}")\n        print(f"RESUME_FROM_COMPLETED_EPOCH={start_epoch}")\n        print(f"NEXT_EPOCH={start_epoch + 1}")\n    checkpoint_dir = args.checkpoint_dir or args.output_dir\n    final_epoch_metrics = None\n    corpus.batch_size = args.batch_size\n    corpus.neg_num = args.neg_num\n    training_range = tqdm(range(start_epoch, args.epochs))\n    for epoch in training_range:\n        model.train()\n        epoch_loss = []\n        epoch_mi_loss = []\n        final_epoch_metrics = None\n        t = time.time()\n        corpus.shuffle()\n        batch_limit = corpus.max_batch_num\n        if args.max_train_batches:\n            batch_limit = min(batch_limit, args.max_train_batches)\n        for batch_num in range(batch_limit):\n            # Training the KGC model\n            if estimator is not None:\n                estimator.eval()\n            optimizer.zero_grad()\n            train_indices, train_values = corpus.get_batch(batch_num)\n            train_indices = torch.LongTensor(train_indices)\n            if args.cuda is not None and int(args.cuda) >= 0:\n                train_indices = train_indices.to(args.device)\n                train_values = train_values.to(args.device)\n            output, embeddings = model.forward(train_indices)\n            loss = model.loss_func(output, train_values)\n            if estimator is not None:\n                loss = loss + args.mu * estimator(embeddings)\n            loss.backward()\n            optimizer.step()\n            # Train the estimator\n            if estimator is not None:\n                estimator.train()\n                optimizer_mi.zero_grad()\n                with torch.no_grad():\n                    embeddings = model.get_batch_embeddings(train_indices)\n                estimator_loss = estimator.train_estimator(embeddings)\n                estimator_loss.backward()\n                optimizer_mi.step()\n            else:\n                estimator_loss = torch.zeros((), device=train_indices.device)\n            epoch_loss.append(loss.data.item())\n            epoch_mi_loss.append(estimator_loss.item())\n        training_range.set_postfix(loss="main: {:.5} mi: {:.5}".format(sum(epoch_loss), sum(epoch_mi_loss)))\n        lr_scheduler.step()\n\n        if (epoch + 1) % args.eval_freq == 0:\n            print("Epoch {:04d} , average loss {:.4f} , epoch_time {:.4f}\\n".format(\n                epoch + 1, sum(epoch_loss) / len(epoch_loss), time.time() - t))\n            model.eval()\n            with torch.no_grad():\n                val_metrics, _ = corpus.get_validation_pred(model, \'test\')\n            final_epoch_metrics = dict(val_metrics)\n            if val_metrics[\'MRR\'] > best_test_metrics[\'MRR\']:\n                best_test_metrics[\'MRR\'] = val_metrics[\'MRR\']\n            if val_metrics[\'MR\'] < best_test_metrics[\'MR\']:\n                best_test_metrics[\'MR\'] = val_metrics[\'MR\']\n            if val_metrics[\'Hits@1\'] > best_test_metrics[\'Hits@1\']:\n                best_test_metrics[\'Hits@1\'] = val_metrics[\'Hits@1\']\n            if val_metrics[\'Hits@3\'] > best_test_metrics[\'Hits@3\']:\n                best_test_metrics[\'Hits@3\'] = val_metrics[\'Hits@3\']\n            if val_metrics[\'Hits@10\'] > best_test_metrics[\'Hits@10\']:\n                best_test_metrics[\'Hits@10\'] = val_metrics[\'Hits@10\']\n            if val_metrics[\'Hits@100\'] > best_test_metrics[\'Hits@100\']:\n                best_test_metrics[\'Hits@100\'] = val_metrics[\'Hits@100\']\n            print(\'\\n\'.join([\'Epoch: {:04d}\'.format(epoch + 1), model.format_metrics(val_metrics, \'test\')]))\n            print("\\n\\n")\n\n        completed_epoch = epoch + 1\n        if checkpoint_dir and (\n            completed_epoch % args.checkpoint_every == 0\n            or final_epoch_metrics is not None\n            or completed_epoch == args.epochs\n        ):\n            payload = build_payload(\n                args,\n                model,\n                estimator,\n                optimizer,\n                optimizer_mi,\n                lr_scheduler,\n                best_test_metrics,\n                corpus,\n                completed_epoch,\n                final_epoch_metrics,\n            )\n            digest = save_checkpoint(Path(checkpoint_dir) / "checkpoint_latest.pt", payload)\n            print(f"CHECKPOINT_SAVED=checkpoint_latest.pt COMPLETED_EPOCH={completed_epoch} SHA256={digest}")\n\n\n    print(\'Total time elapsed: {:.4f}s\'.format(time.time() - t_total))\n    if not best_test_metrics:\n        model.eval()\n        if estimator is not None:\n            estimator.eval()\n        with torch.no_grad():\n            best_test_metrics, _ = corpus.get_validation_pred(model, \'test\')\n    print(\'\\n\'.join([\'Val set results:\', model.format_metrics(best_val_metrics, \'val\')]))\n    print(\'\\n\'.join([\'Test set results:\', model.format_metrics(best_test_metrics, \'test\')]))\n    print("\\n\\n\\n\\n\\n\\n")\n\n    if args.save:\n        if checkpoint_dir:\n            payload = build_payload(\n                args,\n                model,\n                estimator,\n                optimizer,\n                optimizer_mi,\n                lr_scheduler,\n                best_test_metrics,\n                corpus,\n                args.epochs,\n                final_epoch_metrics,\n            )\n            digest = save_checkpoint(Path(checkpoint_dir) / "checkpoint_final.pt", payload)\n            print(f"CHECKPOINT_FINAL_SHA256={digest}")\n        torch.save(model.state_dict(), f\'./checkpoint/{args.dataset}/{args.model}.pth\')\n        print(\'Saved model!\')\n\n\nif __name__ == \'__main__\':\n    train_decoder(args)\n', 'models/RelMoE.py': 'import pickle\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom layers.layer import *\n\nfrom .model import BaseModel\n\n\n\nclass PWLayer(nn.Module):\n    """Single Parametric Whitening Layer\n    """\n    def __init__(self, input_size, output_size, dropout=0.0):\n        super(PWLayer, self).__init__()\n\n        self.dropout = nn.Dropout(p=dropout)\n        self.bias = nn.Parameter(torch.zeros(input_size), requires_grad=True)\n        self.lin = nn.Linear(input_size, output_size, bias=False)\n\n        self.apply(self._init_weights)\n\n    def _init_weights(self, module):\n        if isinstance(module, nn.Linear):\n            module.weight.data.normal_(mean=0.0, std=0.02)\n\n    def forward(self, x):\n        return self.lin(self.dropout(x) - self.bias)\n\n\nclass MoEAdaptorLayer(nn.Module):\n    """MoE-enhanced Adaptor\n    """\n    def __init__(self, n_exps, layers, dropout=0.0, noise=True, use_relation_temperature=True):\n        super(MoEAdaptorLayer, self).__init__()\n\n        self.n_exps = n_exps\n        self.noisy_gating = noise\n        self.use_relation_temperature = use_relation_temperature\n\n        self.experts = nn.ModuleList([PWLayer(layers[0], layers[1], dropout) for i in range(n_exps)])\n        self.w_gate = nn.Parameter(torch.zeros(layers[0], n_exps), requires_grad=True)\n        self.w_noise = nn.Parameter(torch.zeros(layers[0], n_exps), requires_grad=True)\n\n    def noisy_top_k_gating(self, x, r=None, train=None, noise_epsilon=1e-2):\n        clean_logits = x @ self.w_gate\n        if self.noisy_gating and train:\n            raw_noise_stddev = x @ self.w_noise\n            noise_stddev = ((F.softplus(raw_noise_stddev) + noise_epsilon))\n            noisy_logits = clean_logits + (torch.randn_like(clean_logits).to(x.device) * noise_stddev)\n            logits = noisy_logits\n        else:\n            logits = clean_logits\n        if r is not None and self.use_relation_temperature:\n            gates = F.softmax(logits / torch.sigmoid(r), dim=-1)\n        else:\n            gates = F.softmax(logits, dim=-1)\n        return gates\n\n    def forward(self, x, r=None):\n        gates = self.noisy_top_k_gating(x, r, self.training) # (B, n_E)\n        expert_outputs = [self.experts[i](x).unsqueeze(-2) for i in range(self.n_exps)] # [(B, 1, D)]\n        expert_outputs = torch.cat(expert_outputs, dim=-2)\n        multiple_outputs = gates.unsqueeze(-1) * expert_outputs\n        return multiple_outputs.sum(dim=-2), expert_outputs, gates\n    \n\n\nimport pickle\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom layers.layer import *\n\nfrom .model import BaseModel\n\nclass ModalFusionLayer(nn.Module):\n    def __init__(self, in_dim, out_dim, multi, img_dim, txt_dim, fusion_mode="adaptive"):\n        super(ModalFusionLayer, self).__init__()\n\n        self.in_dim = in_dim\n        self.out_dim = out_dim\n        self.multi = multi\n        self.img_dim = img_dim\n        self.text_dim = txt_dim\n        self.fusion_mode = fusion_mode\n\n        modal1 = []\n        for _ in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(in_dim, out_dim)\n            modal1.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal1_layers = nn.ModuleList(modal1)\n\n        modal2 = []\n        for _ in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(self.img_dim, out_dim)\n            modal2.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal2_layers = nn.ModuleList(modal2)\n\n        modal3 = []\n        for _ in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(self.text_dim, out_dim)\n            modal3.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal3_layers = nn.ModuleList(modal3)\n\n        self.ent_attn = nn.Linear(self.out_dim, 1, bias=False)\n        self.ent_attn.requires_grad_(True)\n\n    def forward(self, modal1_emb, modal2_emb, modal3_emb):\n        batch_size = modal1_emb.size(0)\n        x_mm = []\n        for i in range(self.multi):\n            x_modal1 = self.modal1_layers[i](modal1_emb)\n            x_modal2 = self.modal2_layers[i](modal2_emb)\n            x_modal3 = self.modal3_layers[i](modal3_emb)\n            x_stack = torch.stack((x_modal1, x_modal2, x_modal3), dim=1)\n            attention_scores = self.ent_attn(x_stack).squeeze(-1)\n            if self.fusion_mode == "equal":\n                attention_weights = torch.full_like(attention_scores, 1.0 / 3.0)\n            else:\n                attention_weights = torch.softmax(attention_scores, dim=-1)\n            context_vectors = torch.sum(attention_weights.unsqueeze(-1) * x_stack, dim=1)\n            x_mm.append(context_vectors)\n        x_mm = torch.stack(x_mm, dim=1)\n        x_mm = x_mm.sum(1).view(batch_size, self.out_dim)\n        # x_mm = torch.relu(x_mm)\n        return x_mm, attention_weights\n    \n    def relation_gated_fuse(self, modal1_emb, modal2_emb, modal3_emb, rel):\n        batch_size = modal1_emb.size(0)\n        x_mm = []\n        for i in range(self.multi):\n            x_modal1 = self.modal1_layers[i](modal1_emb)\n            x_modal2 = self.modal2_layers[i](modal2_emb)\n            x_modal3 = self.modal3_layers[i](modal3_emb)\n            x_stack = torch.stack((x_modal1, x_modal2, x_modal3), dim=1)\n            attention_scores = self.ent_attn(x_stack).squeeze(-1)\n            attention_weights = torch.softmax(attention_scores / rel, dim=-1)\n            context_vectors = torch.sum(attention_weights.unsqueeze(-1) * x_stack, dim=1)\n            x_mm.append(context_vectors)\n        x_mm = torch.stack(x_mm, dim=1)\n        x_mm = x_mm.mean(1).view(batch_size, self.out_dim)\n        x_mm = torch.relu(x_mm)\n        return x_mm\n    \n    def gated_fusion(self, emb, rel):\n        # emb: batch_size x dim\n        # rel: batch_size x dim\n        w = torch.sigmoid(emb * rel)\n        return w * emb + (1 - w) * rel\n\n\n\nclass RelMoE(BaseModel):\n    def __init__(self, args):\n        super(RelMoE, self).__init__(args)\n        variant = getattr(args, "variant_config", None)\n        self.variant_config = variant\n        self.training_head_names = tuple(\n            getattr(variant, "active_heads", ("structure", "image", "text", "joint"))\n        )\n        if variant is not None and not variant.train_joint_head and "joint" in self.training_head_names:\n            self.training_head_names = tuple(\n                head for head in self.training_head_names if head != "joint"\n            )\n        self.evaluation_head_indices = tuple(\n            {"structure": 0, "image": 1, "text": 2, "joint": 3}[head]\n            for head in getattr(variant, "active_heads", ("structure", "image", "text", "joint"))\n            if getattr(variant, "infer_joint_head", True) or head != "joint"\n        )\n        use_noise = getattr(variant, "use_gating_noise", True)\n        use_temperature = getattr(variant, "use_relation_temperature", True)\n        fusion_mode = getattr(variant, "fusion_mode", "adaptive")\n        self.entity_embeddings = nn.Embedding(\n            len(args.entity2id),\n            args.dim,\n            padding_idx=None\n        )\n        nn.init.xavier_normal_(self.entity_embeddings.weight)\n\n        self.relation_embeddings = nn.Embedding(\n            2 * len(args.relation2id), \n            args.r_dim, \n            padding_idx=None\n        )\n        nn.init.xavier_normal_(self.relation_embeddings.weight)\n\n        if args.pre_trained:\n            self.entity_embeddings = nn.Embedding.from_pretrained(\n                torch.from_numpy(pickle.load(open(\'datasets/\' + args.dataset + \'/gat_entity_vec.pkl\', \'rb\'))).float(), freeze=False)\n            self.relation_embeddings = nn.Embedding.from_pretrained(torch.cat((\n                torch.from_numpy(pickle.load(open(\'datasets/\' + args.dataset + \'/gat_relation_vec.pkl\', \'rb\'))).float(),\n                -1 * torch.from_numpy(pickle.load(open(\'datasets/\' + args.dataset + \'/gat_relation_vec.pkl\', \'rb\'))).float()), dim=0), freeze=False)\n\n        self.rel_gate = nn.Embedding(2 * len(args.relation2id), 1, padding_idx=None)\n\n        if args.dataset == "DB15K":\n            img_pool = torch.nn.AvgPool2d(4, stride=4)\n            img = img_pool(args.img.to(self.device).view(-1, 64, 64))\n            img = img.view(img.size(0), -1)\n            txt_pool = torch.nn.AdaptiveAvgPool2d(output_size=(4, 64))\n            txt = txt_pool(args.desp.to(self.device).view(-1, 12, 64))\n            txt = txt.view(txt.size(0), -1)\n        elif "MKG" in args.dataset:\n            # multi-modal information for MKG\n            img = args.img.to(self.device).view(args.img.size(0), -1)\n            txt_pool = torch.nn.AdaptiveAvgPool2d(output_size=(4, 64))\n            txt = txt_pool(args.desp.to(self.device).view(-1, 12, 32))\n            txt = txt.view(txt.size(0), -1)\n        elif "TIVA" in args.dataset:\n            img_pool = torch.nn.AdaptiveAvgPool2d(output_size=(4, 64))\n            img = img_pool(args.img.to(self.device).view(-1, 32, 64))\n            img = img.view(img.size(0), -1)\n            txt = args.desp.to(self.device)\n            txt = txt.view(txt.size(0), -1)\n        elif "Kuai" in args.dataset:\n            img_pool = torch.nn.AdaptiveAvgPool2d(output_size=(4, 64))\n            img = img_pool(args.img.to(self.device).view(-1, 12, 64))\n            img = img.view(img.size(0), -1)\n            txt_pool = torch.nn.AdaptiveAvgPool2d(output_size=(4, 64))\n            txt = txt_pool(args.desp.to(self.device).view(-1, 12, 64))\n            txt = txt.view(txt.size(0), -1)\n        elif "WN9" in args.dataset:\n            img_pool = torch.nn.AvgPool2d(4, stride=4)\n            img = img_pool(args.img.to(self.device).view(-1, 64, 64))\n            img = img.view(img.size(0), -1)\n            img = torch.tensor(img).to(torch.float32)\n            txt = args.desp.to(self.device)\n            txt = txt.view(txt.size(0), -1)\n            txt = torch.tensor(txt).to(torch.float32)\n        elif "FB15K-237" in args.dataset:\n            img_pool = torch.nn.AdaptiveAvgPool2d(output_size=(4, 64))\n            img = img_pool(args.img.to(self.device).view(-1, 12, 64))\n            img = img.view(img.size(0), -1)\n            txt_pool = torch.nn.AdaptiveAvgPool2d(output_size=(4, 64))\n            txt = txt_pool(args.desp.to(self.device).view(-1, 12, 64))\n            txt = txt.view(txt.size(0), -1)\n\n        self.img_entity_embeddings = nn.Embedding.from_pretrained(img, freeze=False)\n        self.img_relation_embeddings = nn.Embedding(\n            2 * len(args.relation2id),\n            args.r_dim, \n            padding_idx=None\n        )\n        nn.init.xavier_normal_(self.img_relation_embeddings.weight)\n        self.txt_entity_embeddings = nn.Embedding.from_pretrained(txt, freeze=False)\n        self.txt_relation_embeddings = nn.Embedding(\n            2 * len(args.relation2id),\n            args.r_dim,\n            padding_idx=None\n        )\n        nn.init.xavier_normal_(self.txt_relation_embeddings.weight)\n\n        # Score Functions\n        self.dim = args.dim\n        self.img_dim = self.img_entity_embeddings.weight.data.shape[1]\n        self.txt_dim = self.txt_entity_embeddings.weight.data.shape[1]\n        self.fuse_out_dim = self.dim\n        # Score function layers\n        self.TuckER_S = TuckERLayer(args.dim, args.r_dim)\n        self.TuckER_I = TuckERLayer(self.img_dim, args.r_dim)\n        self.TuckER_D = TuckERLayer(self.txt_dim, args.r_dim)\n        self.TuckER_MM = TuckERLayer(args.dim, self.fuse_out_dim)\n        # Multi-modal fusion layers\n\n        self.visual_moe = MoEAdaptorLayer(n_exps=args.n_exp, layers=[self.img_dim, self.img_dim], noise=use_noise, use_relation_temperature=use_temperature)\n        self.text_moe = MoEAdaptorLayer(n_exps=args.n_exp, layers=[self.txt_dim, self.txt_dim], noise=use_noise, use_relation_temperature=use_temperature)\n        self.structure_moe = MoEAdaptorLayer(n_exps=args.n_exp, layers=[self.dim, self.dim], noise=use_noise, use_relation_temperature=use_temperature)\n        self.mm_moe = MoEAdaptorLayer(n_exps=args.n_exp, layers=[self.fuse_out_dim, self.fuse_out_dim], noise=use_noise, use_relation_temperature=use_temperature)\n    \n        self.fuse_e = ModalFusionLayer(\n            in_dim=args.dim,\n            out_dim=self.fuse_out_dim,\n            multi=2,\n            img_dim=self.img_dim,\n            txt_dim=self.txt_dim,\n            fusion_mode=fusion_mode\n        )\n        self.fuse_r = ModalFusionLayer(\n            in_dim=args.r_dim,\n            out_dim=self.fuse_out_dim,\n            multi=2,\n            img_dim=args.r_dim,\n            txt_dim=args.r_dim,\n            fusion_mode=fusion_mode\n        )\n        self.bias = nn.Parameter(torch.zeros(len(args.entity2id)))\n        self.bceloss = nn.BCELoss()\n\n        \n    def forward(self, batch_inputs):\n        head = batch_inputs[:, 0]\n        relation = batch_inputs[:, 1]\n        rel_gate = self.rel_gate(relation)\n        e_embed, disen_str, atten_s = self.structure_moe(self.entity_embeddings(head), rel_gate)\n        r_embed = self.relation_embeddings(relation)\n        e_img_embed, disen_img, atten_i = self.visual_moe(self.img_entity_embeddings(head), rel_gate)\n        r_img_embed = self.img_relation_embeddings(relation)\n        e_txt_embed, disen_txt, atten_t = self.text_moe(self.txt_entity_embeddings(head), rel_gate)\n        r_txt_embed = self.txt_relation_embeddings(relation)\n        e_mm_embed, attn_f = self.fuse_e(e_embed, e_img_embed, e_txt_embed)\n        r_mm_embed, _ = self.fuse_r(r_embed, r_img_embed, r_txt_embed)\n        \n        pred_s = self.TuckER_S(e_embed, r_embed)\n        pred_i = self.TuckER_I(e_img_embed, r_img_embed)\n        pred_d = self.TuckER_D(e_txt_embed, r_txt_embed)\n        pred_mm = self.TuckER_MM(e_mm_embed, r_mm_embed)\n        \n        all_s, _, _ = self.structure_moe(self.entity_embeddings.weight)\n        all_v, _, _ = self.visual_moe(self.img_entity_embeddings.weight)\n        all_t, _, _ = self.text_moe(self.txt_entity_embeddings.weight)\n        all_f, _ = self.fuse_e(all_s, all_v, all_t)\n        pred_s = torch.mm(pred_s, all_s.transpose(1, 0))\n        pred_i = torch.mm(pred_i, all_v.transpose(1, 0))\n        pred_d = torch.mm(pred_d, all_t.transpose(1, 0))\n        pred_mm = torch.mm(pred_mm, all_f.transpose(1, 0))\n\n        pred_s = torch.sigmoid(pred_s)\n        pred_i = torch.sigmoid(pred_i)\n        pred_d = torch.sigmoid(pred_d)\n        pred_mm = torch.sigmoid(pred_mm)\n        if not self.training:\n            return [pred_s, pred_i, pred_d, pred_mm], [atten_s, atten_i, atten_t, attn_f]\n        else:\n            return [pred_s, pred_i, pred_d, pred_mm], [disen_str, disen_img, disen_txt]\n    \n    def get_batch_embeddings(self, batch_inputs):\n        head = batch_inputs[:, 0]\n        _, disen_str, _ = self.structure_moe(self.entity_embeddings(head))\n        _, disen_img, _ = self.visual_moe(self.img_entity_embeddings(head))\n        _, disen_txt, _ = self.text_moe(self.txt_entity_embeddings(head))\n        return [disen_str, disen_img, disen_txt]\n\n\n    def loss_func(self, output, target):\n        head_indices = {"structure": 0, "image": 1, "text": 2, "joint": 3}\n        losses = [\n            self.bceloss(output[head_indices[name]], target)\n            for name in self.training_head_names\n        ]\n        if not losses:\n            raise RuntimeError("variant has no supervised prediction heads")\n        return sum(losses)\n', 'models/modules.py': 'import torch\nimport torch.nn as nn\nimport random\n\n\n\nclass Similarity(nn.Module):\n    """\n    Dot product or cosine similarity\n    """\n\n    def __init__(self, temp):\n        super().__init__()\n        self.temp = temp\n        self.cos = nn.CosineSimilarity(dim=-1)\n\n    def forward(self, x, y):\n        return self.cos(x, y) / self.temp\n    \nclass ContrastiveLoss(nn.Module):\n    def __init__(self, temp=0.5):\n        super().__init__()\n        self.loss = nn.CrossEntropyLoss()\n        self.sim_func = Similarity(temp=temp)\n\n    def forward(self, emb1, emb2):\n        batch_sim = self.sim_func(emb1.unsqueeze(1), emb2.unsqueeze(0))\n        labels = torch.arange(batch_sim.size(0)).long().to(\'cuda\')\n        return self.loss(batch_sim, labels)\n\n\nclass CLUBSample(nn.Module):  # Sampled version of the CLUB estimator\n    def __init__(self, x_dim, y_dim, hidden_size):\n        super(CLUBSample, self).__init__()\n        self.p_mu = nn.Sequential(\n            nn.Linear(x_dim, hidden_size//2),\n            nn.ReLU(),\n            nn.Linear(hidden_size//2, y_dim)\n        )\n\n        self.p_logvar = nn.Sequential(\n            nn.Linear(x_dim, hidden_size//2),\n            nn.ReLU(),\n            nn.Linear(hidden_size//2, y_dim),\n            nn.Tanh()\n        )\n\n    def get_mu_logvar(self, x_samples):\n        mu = self.p_mu(x_samples)\n        logvar = self.p_logvar(x_samples)\n        return mu, logvar\n     \n        \n    def loglikeli(self, x_samples, y_samples):\n        mu, logvar = self.get_mu_logvar(x_samples)\n        return (-(mu - y_samples)**2 /2./logvar.exp()).sum(dim=1).mean(dim=0)\n    \n\n    def forward(self, x_samples, y_samples):\n        mu, logvar = self.get_mu_logvar(x_samples)\n        \n        sample_size = x_samples.shape[0]\n        #random_index = torch.randint(sample_size, (sample_size,)).long()\n        random_index = torch.randperm(sample_size).long()\n        \n        positive = - (mu - y_samples)**2 / logvar.exp()\n        negative = - (mu - y_samples[random_index])**2 / logvar.exp()\n        upper_bound = (positive.sum(dim = -1) - negative.sum(dim = -1)).mean()\n        return upper_bound / 2.0\n\n    def learning_loss(self, x_samples, y_samples):\n        return - self.loglikeli(x_samples, y_samples)\n\n\nclass MIEstimator(nn.Module):\n    def __init__(self, args):\n        super(MIEstimator, self).__init__()\n        self.str_estimator = CLUBSample(args.dim, args.dim, args.dim)\n        self.img_estimator = CLUBSample(args.img_dim, args.img_dim, args.img_dim)\n        self.txt_estimator = CLUBSample(args.txt_dim, args.txt_dim, args.txt_dim)\n        self.num = args.n_exp\n    \n\n    def forward(self, embeddings):\n        strs, imgs, txts = embeddings\n        bzs, n_exp, _ = strs.size()\n        assert n_exp == self.num\n        idx1, idx2 = random.sample(range(n_exp), k=2)\n        str1, str2 = strs[:, idx1, :], strs[:, idx2, :]\n        img1, img2 = imgs[:, idx1, :], imgs[:, idx2, :]\n        txt1, txt2 = txts[:, idx1, :], txts[:, idx2, :]\n        mi_loss = (\n            self.str_estimator(str1, str2)\n            + self.img_estimator(img1, img2)\n            + self.txt_estimator(txt1, txt2)\n        ) / 3.0\n        return mi_loss\n\n    def train_estimator(self, embeddings):\n        strs, imgs, txts = embeddings\n        bzs, n_exp, _ = strs.size()\n        assert n_exp == self.num\n        idx1, idx2 = random.sample(range(n_exp), k=2)\n        str1, str2 = strs[:, idx1, :], strs[:, idx2, :]\n        img1, img2 = imgs[:, idx1, :], imgs[:, idx2, :]\n        txt1, txt2 = txts[:, idx1, :], txts[:, idx2, :]\n        est_loss = (\n            self.str_estimator.learning_loss(str1, str2)\n            + self.img_estimator.learning_loss(img1, img2)\n            + self.txt_estimator.learning_loss(txt1, txt2)\n        ) / 3.0\n        return est_loss\n', 'models/model.py': 'import numpy as np\nimport pickle\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom layers.layer import *\n\n\nclass BaseModel(nn.Module):\n    def __init__(self, args):\n        super(BaseModel, self).__init__()\n        self.device = args.device\n\n    @staticmethod\n    def format_metrics(metrics, split):\n        return " ".join(\n            ["{}_{}: {:.4f}".format(split, metric_name, metric_val) for metric_name, metric_val in metrics.items()])\n\n    @staticmethod\n    def has_improved(m1, m2):\n        return (m1["Mean Rank"] > m2["Mean Rank"]) or (m1["Mean Reciprocal Rank"] < m2["Mean Reciprocal Rank"])\n\n    @staticmethod\n    def init_metric_dict():\n        return {"Hits@100": -1, "Hits@10": -1, "Hits@3": -1, "Hits@1": -1,\n                "MR": 100000, "MRR": -1}\n\n\nclass Mutan(BaseModel):\n    def __init__(self, args):\n        super(Mutan, self).__init__(args)\n        self.entity_embeddings = nn.Embedding(len(args.entity2id), args.dim, padding_idx=None)\n        nn.init.xavier_normal_(self.entity_embeddings.weight)\n        self.relation_embeddings = nn.Embedding(2 * len(args.relation2id), args.r_dim, padding_idx=None)\n        nn.init.xavier_normal_(self.relation_embeddings.weight)\n        self.dim = args.dim\n        self.Mutan = MutanLayer(args.dim, 5)\n        self.bias = nn.Parameter(torch.zeros(len(args.entity2id)))\n        self.bceloss = nn.BCELoss()\n\n    def forward(self, batch_inputs):\n        head = batch_inputs[:, 0]\n        relation = batch_inputs[:, 1]\n        e_embed = self.entity_embeddings(head)\n        r_embed = self.relation_embeddings(relation)\n        pred = self.Mutan(e_embed, r_embed)\n        pred = torch.mm(pred, self.entity_embeddings.weight.transpose(1, 0))\n        pred = torch.sigmoid(pred)\n        return pred\n\n    def loss_func(self, output, target):\n        return self.bceloss(output, target)\n\n\nclass TuckER(BaseModel):\n    def __init__(self, args):\n        super(TuckER, self).__init__(args)\n        self.entity_embeddings = nn.Embedding(len(args.entity2id), args.dim, padding_idx=None)\n        nn.init.xavier_normal_(self.entity_embeddings.weight)\n        self.relation_embeddings = nn.Embedding(2 * len(args.relation2id), args.r_dim, padding_idx=None)\n        nn.init.xavier_normal_(self.relation_embeddings.weight)\n        if args.pre_trained:\n            self.entity_embeddings = nn.Embedding.from_pretrained(\n                torch.from_numpy(pickle.load(open(\'datasets/\' + args.dataset + \'/gat_entity_vec.pkl\', \'rb\'))).float(), freeze=False)\n            self.relation_embeddings = nn.Embedding.from_pretrained(torch.cat((\n                torch.from_numpy(pickle.load(open(\'datasets/\' + args.dataset + \'/gat_relation_vec.pkl\', \'rb\'))).float(),\n                -1 * torch.from_numpy(pickle.load(open(\'datasets/\' + args.dataset + \'/gat_relation_vec.pkl\', \'rb\'))).float()), dim=0), freeze=False)\n        self.dim = args.dim\n        self.TuckER = TuckERLayer(args.dim, args.r_dim)\n        self.bias = nn.Parameter(torch.zeros(len(args.entity2id)))\n        self.bceloss = nn.BCELoss()\n\n    def forward(self, batch_inputs, lookup=None):\n        head = batch_inputs[:, 0]\n        relation = batch_inputs[:, 1]\n        e_embed = self.entity_embeddings(head)\n        r_embed = self.relation_embeddings(relation)\n        pred = self.TuckER(e_embed, r_embed)\n        if lookup is None:\n            pred = torch.mm(pred, self.entity_embeddings.weight.transpose(1, 0))\n        else:\n            pred = torch.bmm(pred.unsqueeze(1), self.entity_embeddings.weight[lookup].transpose(1, 2)).squeeze(1)\n        pred = torch.sigmoid(pred)\n        return pred\n\n    def loss_func(self, output, target):\n        return self.bceloss(output, target)\n\n\nclass ConvE(BaseModel):\n    def __init__(self, args):\n        super(ConvE, self).__init__(args)\n        self.entity_embeddings = nn.Embedding(len(args.entity2id), args.dim, padding_idx=None)\n        nn.init.xavier_normal_(self.entity_embeddings.weight)\n        self.relation_embeddings = nn.Embedding(2 * len(args.relation2id), args.dim, padding_idx=None)\n        nn.init.xavier_normal_(self.relation_embeddings.weight)\n        self.dim = args.dim\n        self.k_w = args.k_w\n        self.k_h = args.k_h\n        self.ConvE = ConvELayer(args.dim, args.out_channels, args.kernel_size, args.k_h, args.k_w)\n        self.bias = nn.Parameter(torch.zeros(len(args.entity2id)))\n        self.bceloss = nn.BCELoss()\n\n    def forward(self, batch_inputs):\n        head = batch_inputs[:, 0]\n        relation = batch_inputs[:, 1]\n        e_embed = self.entity_embeddings(head)\n        r_embed = self.relation_embeddings(relation)\n        e_embed = e_embed.view(-1, 1, self.dim)\n        r_embed = r_embed.view(-1, 1, self.dim)\n        embed = torch.cat([e_embed, r_embed], dim=1)\n        embed = torch.transpose(embed, 2, 1).reshape((-1, 1, 2 * self.k_w, self.k_h))\n\n        pred = self.ConvE(embed)\n        pred = torch.mm(pred, self.entity_embeddings.weight.transpose(1, 0))\n        pred += self.bias.expand_as(pred)\n        pred = torch.sigmoid(pred)\n        return pred\n\n    def loss_func(self, output, target):\n        return self.bceloss(output, target)\n\n\n\n', 'utils/data_loader.py': 'import torch\nimport numpy as np\nfrom collections import defaultdict\n\n\nclass Corpus:\n    def __init__(self, args, train_data, val_data, test_data, entity2id, relation2id):\n        self.device = args.device\n        self.train_triples = train_data[0]\n        self.val_triples = val_data[0]\n        self.test_triples = test_data[0]\n        self.max_batch_num = 1\n\n        adj_indices = torch.LongTensor([train_data[1][0], train_data[1][1]])\n        adj_values = torch.LongTensor([train_data[1][2]])\n        self.train_adj_matrix = (adj_indices, adj_values)\n\n        self.entity2id = {k: v for k, v in entity2id.items()}\n        self.id2entity = {v: k for k, v in entity2id.items()}\n        self.relation2id = {k: v for k, v in relation2id.items()}\n        self.id2relation = {v: k for k, v in relation2id.items()}\n        self.batch_size = args.batch_size\n\n    def shuffle(self):\n        raise NotImplementedError\n\n    def get_batch(self, batch_num):\n        raise NotImplementedError\n\n    def get_validation_pred(self, model, split=\'test\'):\n        raise NotImplementedError\n\n\nclass ConvECorpus(Corpus):\n    def __init__(self, args, train_data, val_data, test_data, entity2id, relation2id):\n        super(ConvECorpus, self).__init__(args, train_data, val_data, test_data, entity2id, relation2id)\n        rel_num = len(relation2id)\n        for k, v in relation2id.items():\n            self.relation2id[k+\'_reverse\'] = v+rel_num\n        self.id2relation = {v: k for k, v in self.relation2id.items()}\n\n        sr2o = {}\n        for (head, relation, tail) in self.train_triples:\n            if (head, relation) not in sr2o.keys():\n                sr2o[(head, relation)] = set()\n            if (tail, relation+rel_num) not in sr2o.keys():\n                sr2o[(tail, relation+rel_num)] = set()\n            sr2o[(head, relation)].add(tail)\n            sr2o[(tail, relation+rel_num)].add(head)\n\n        self.triples = {}\n        self.train_indices = [{\'triple\': (head, relation, -1), \'label\': list(sr2o[(head, relation)])}\n                              for (head, relation), tail in sr2o.items()]\n        self.triples[\'train\'] = [{\'triple\': (head, relation, -1), \'label\': list(sr2o[(head, relation)])}\n                                 for (head, relation), tail in sr2o.items()]\n\n        if len(self.train_indices) % self.batch_size == 0:\n            self.max_batch_num = len(self.train_indices) // self.batch_size\n        else:\n            self.max_batch_num = len(self.train_indices) // self.batch_size + 1\n\n        for (head, relation, tail) in self.val_triples:\n            if (head, relation) not in sr2o.keys():\n                sr2o[(head, relation)] = set()\n            if (tail, relation+rel_num) not in sr2o.keys():\n                sr2o[(tail, relation+rel_num)] = set()\n            sr2o[(head, relation)].add(tail)\n            sr2o[(tail, relation+rel_num)].add(head)\n\n        for (head, relation, tail) in self.test_triples:\n            if (head, relation) not in sr2o.keys():\n                sr2o[(head, relation)] = set()\n            if (tail, relation+rel_num) not in sr2o.keys():\n                sr2o[(tail, relation+rel_num)] = set()\n            sr2o[(head, relation)].add(tail)\n            sr2o[(tail, relation+rel_num)].add(head)\n\n        self.val_head_indices = [{\'triple\': (tail, relation + rel_num, head), \'label\': list(sr2o[(tail, relation + rel_num)])}\n                                 for (head, relation, tail) in self.val_triples]\n        self.val_tail_indices = [{\'triple\': (head, relation, tail), \'label\': list(sr2o[(head, relation)])}\n                                 for (head, relation, tail) in self.val_triples]\n        self.test_head_indices = [{\'triple\': (tail, relation + rel_num, head), \'label\': list(sr2o[(tail, relation + rel_num)])}\n                                 for (head, relation, tail) in self.test_triples]\n        self.test_tail_indices = [{\'triple\': (head, relation, tail), \'label\': list(sr2o[(head, relation)])}\n                                 for (head, relation, tail) in self.test_triples]\n\n\n    def read_batch(self, batch):\n        triple, label = [_.to(self.device) for _ in batch]\n        return triple, label\n\n    def shuffle(self):\n        np.random.shuffle(self.train_indices)\n\n    def get_batch(self, batch_num):\n        if (batch_num + 1) * self.batch_size <= len(self.train_indices):\n            batch = self.train_indices[batch_num * self.batch_size: (batch_num+1) * self.batch_size]\n        else:\n            batch = self.train_indices[batch_num * self.batch_size:]\n        batch_indices = torch.LongTensor([indice[\'triple\'] for indice in batch])\n        label = [np.int32(indice[\'label\']) for indice in batch]\n        y = np.zeros((len(batch), len(self.entity2id)), dtype=np.float32)\n        for idx in range(len(label)):\n            for l in label[idx]:\n                y[idx][l] = 1.0\n        y = 0.9 * y + (1.0 / len(self.entity2id))\n        batch_values = torch.FloatTensor(y)\n\n        \'\'\'index = []\n        for idx in range(len(label)):\n            pos = label[idx]\n            np.random.shuffle(pos)\n            neg = np.int32(list(range((len(self.entity2id)))))\n            np.random.shuffle(neg)\n            if len(pos) >= 10:\n                index.append(np.concatenate((pos[:10], neg[:90])))\n            else:\n                index.append(np.concatenate((pos, neg[:100-len(pos)])))\n        y = torch.FloatTensor(y)\n        index = torch.LongTensor(index)\n        batch_values = torch.gather(y, dim=1, index=index)\'\'\'\n\n        return batch_indices, batch_values#, index\n\n    def get_validation_pred(self, model, split=\'test\'):\n        ranks_head, ranks_tail = [], []\n        reciprocal_ranks_head, reciprocal_ranks_tail = [], []\n        hits_at_100_head, hits_at_100_tail = 0, 0\n        hits_at_10_head, hits_at_10_tail = 0, 0\n        hits_at_3_head, hits_at_3_tail = 0, 0\n        hits_at_1_head, hits_at_1_tail = 0, 0\n\n        rel_pred_dict = defaultdict(list)\n        att_s = []\n        att_i = []\n        att_t = []\n        att_mm = []\n\n        if split == \'val\':\n            head_indices = self.val_head_indices\n            tail_indices = self.val_tail_indices\n        else:\n            head_indices = self.test_head_indices\n            tail_indices = self.test_tail_indices\n\n        if len(head_indices) % self.batch_size == 0:\n            max_batch_num = len(head_indices) // self.batch_size\n        else:\n            max_batch_num = len(head_indices) // self.batch_size + 1\n        for batch_num in range(max_batch_num):\n            if (batch_num + 1) * self.batch_size <= len(head_indices):\n                head_batch = head_indices[batch_num * self.batch_size: (batch_num + 1) * self.batch_size]\n                tail_batch = tail_indices[batch_num * self.batch_size: (batch_num + 1) * self.batch_size]\n            else:\n                head_batch = head_indices[batch_num * self.batch_size:]\n                tail_batch = tail_indices[batch_num * self.batch_size:]\n\n            head_batch_indices = torch.LongTensor([indice[\'triple\'] for indice in head_batch])\n            head_batch_indices = head_batch_indices.to(self.device)\n            rel_ids = head_batch_indices[:, 1]\n            pred, attention = model.forward(head_batch_indices)\n            for i in range(pred[0].shape[0]):\n                h, r, t = head_batch_indices[i][0].item(), head_batch_indices[i][1].item(), head_batch_indices[i][2].item()\n                atts = attention[0][i]\n                atti = attention[1][i]\n                attt = attention[2][i]\n                attmm = attention[3][i]\n                att_s.append((h, r, t, atts))\n                att_i.append((h, r, t, atti))\n                att_t.append((h, r, t, attt))\n                att_mm.append((h, r, t, attmm))\n            selected_heads = getattr(model, \'evaluation_head_indices\', (0, 1, 2, 3))\n            pred = sum(pred[index] for index in selected_heads) / len(selected_heads)\n            label = [np.int32(indice[\'label\']) for indice in head_batch]\n            y = np.zeros((len(head_batch), len(self.entity2id)), dtype=np.float32)\n            for idx in range(len(label)):\n                for l in label[idx]:\n                    y[idx][l] = 1.0\n            y = torch.FloatTensor(y).to(self.device)\n            target = head_batch_indices[:, 2]\n            b_range = torch.arange(pred.shape[0], device=self.device)\n            target_pred = pred[b_range, target]\n            pred = torch.where(y.byte(), torch.zeros_like(pred), pred)\n            pred[b_range, target] = target_pred\n            pred = pred.cpu().numpy()\n            target = target.cpu().numpy()\n            # pred.shape[0] = batch_size\n            for i in range(pred.shape[0]):\n                scores = pred[i]\n                tar = target[i]\n                tar_scr = scores[tar]\n                scores = np.delete(scores, tar)\n                rand = np.random.randint(scores.shape[0])\n                scores = np.insert(scores, rand, tar_scr)\n                sorted_indices = np.argsort(-scores, kind=\'stable\')\n                # higher is better\n                ranks_head.append(np.where(sorted_indices == rand)[0][0]+1)\n                reciprocal_ranks_head.append(1.0 / ranks_head[-1])\n                rel_pred_dict[rel_ids[i].item()].append(ranks_head[-1])\n\n            tail_batch_indices = torch.LongTensor([indice[\'triple\'] for indice in tail_batch])\n            tail_batch_indices = tail_batch_indices.to(self.device)\n            rel_ids = tail_batch_indices[:, 1]\n            pred, attention = model.forward(tail_batch_indices)\n            for i in range(pred[0].shape[0]):\n                h, r, t = tail_batch_indices[i][0].item(), tail_batch_indices[i][1].item(), tail_batch_indices[i][2].item()\n                atts = attention[0][i]\n                atti = attention[1][i]\n                attt = attention[2][i]\n                attmm = attention[3][i]\n                att_s.append((h, r, t, atts))\n                att_i.append((h, r, t, atti))\n                att_t.append((h, r, t, attt))\n                att_mm.append((h, r, t, attmm))\n            selected_heads = getattr(model, \'evaluation_head_indices\', (0, 1, 2, 3))\n            pred = sum(pred[index] for index in selected_heads) / len(selected_heads)\n            label = [np.int32(indice[\'label\']) for indice in tail_batch]\n            y = np.zeros((len(tail_batch), len(self.entity2id)), dtype=np.float32)\n            for idx in range(len(label)):\n                for l in label[idx]:\n                    y[idx][l] = 1.0\n            y = torch.FloatTensor(y).to(self.device)\n            target = tail_batch_indices[:, 2]\n            b_range = torch.arange(pred.shape[0], device=self.device)\n            target_pred = pred[b_range, target]\n            pred = torch.where(y.byte(), torch.zeros_like(pred), pred)\n            pred[b_range, target] = target_pred\n            pred = pred.cpu().numpy()\n            target = target.cpu().numpy()\n            for i in range(pred.shape[0]):\n                scores = pred[i]\n                tar = target[i]\n                tar_scr = scores[tar]\n                scores = np.delete(scores, tar)\n                rand = np.random.randint(scores.shape[0])\n                scores = np.insert(scores, rand, tar_scr)\n                sorted_indices = np.argsort(-scores, kind=\'stable\')\n                ranks_tail.append(np.where(sorted_indices == rand)[0][0] + 1)\n                reciprocal_ranks_tail.append(1.0 / ranks_tail[-1])\n                rel_pred_dict[rel_ids[i].item()].append(ranks_head[-1])\n\n        for i in range(len(ranks_head)):\n            if ranks_head[i] <= 100:\n                hits_at_100_head += 1\n            if ranks_head[i] <= 10:\n                hits_at_10_head += 1\n            if ranks_head[i] <= 3:\n                hits_at_3_head += 1\n            if ranks_head[i] == 1:\n                hits_at_1_head += 1\n\n        for i in range(len(ranks_tail)):\n            if ranks_tail[i] <= 100:\n                hits_at_100_tail += 1\n            if ranks_tail[i] <= 10:\n                hits_at_10_tail += 1\n            if ranks_tail[i] <= 3:\n                hits_at_3_tail += 1\n            if ranks_tail[i] == 1:\n                hits_at_1_tail += 1\n\n        assert len(ranks_head) == len(reciprocal_ranks_head)\n        assert len(ranks_tail) == len(reciprocal_ranks_tail)\n\n        hits_100_head = hits_at_100_head / len(ranks_head)\n        hits_10_head = hits_at_10_head / len(ranks_head)\n        hits_3_head = hits_at_3_head / len(ranks_head)\n        hits_1_head = hits_at_1_head / len(ranks_head)\n        mean_rank_head = sum(ranks_head) / len(ranks_head)\n        mean_reciprocal_rank_head = sum(reciprocal_ranks_head) / len(reciprocal_ranks_head)\n\n        hits_100_tail = hits_at_100_tail / len(ranks_tail)\n        hits_10_tail = hits_at_10_tail / len(ranks_tail)\n        hits_3_tail = hits_at_3_tail / len(ranks_tail)\n        hits_1_tail = hits_at_1_tail / len(ranks_tail)\n        mean_rank_tail = sum(ranks_tail) / len(ranks_tail)\n        mean_reciprocal_rank_tail = sum(reciprocal_ranks_tail) / len(reciprocal_ranks_tail)\n\n        hits_100 = (hits_at_100_head / len(ranks_head) + hits_at_100_tail / len(ranks_tail)) / 2\n        hits_10 = (hits_at_10_head / len(ranks_head) + hits_at_10_tail / len(ranks_tail)) / 2\n        hits_3 = (hits_at_3_head / len(ranks_head) + hits_at_3_tail / len(ranks_tail)) / 2\n        hits_1 = (hits_at_1_head / len(ranks_head) + hits_at_1_tail / len(ranks_tail)) / 2\n        mean_rank = (sum(ranks_head) / len(ranks_head) + sum(ranks_tail) / len(ranks_tail)) / 2\n        mean_reciprocal_rank = (sum(reciprocal_ranks_head) / len(reciprocal_ranks_head) + sum(\n            reciprocal_ranks_tail) / len(reciprocal_ranks_tail)) / 2\n\n        metrics = {\n            "Hits@100": hits_100,\n            "Hits@10": hits_10,\n            "Hits@3": hits_3,\n            "Hits@1": hits_1,\n            "MR": mean_rank,\n            "MRR": mean_reciprocal_rank\n        }\n\n        return metrics, [att_s, att_i, att_t, att_mm]\n    \n    def get_validation_pred_signle(self, model, split=\'test\', index=0):\n        ranks_head, ranks_tail = [], []\n        reciprocal_ranks_head, reciprocal_ranks_tail = [], []\n        hits_at_100_head, hits_at_100_tail = 0, 0\n        hits_at_10_head, hits_at_10_tail = 0, 0\n        hits_at_3_head, hits_at_3_tail = 0, 0\n        hits_at_1_head, hits_at_1_tail = 0, 0\n        rel_pred_dict = defaultdict(list)\n\n        if split == \'val\':\n            head_indices = self.val_head_indices\n            tail_indices = self.val_tail_indices\n        else:\n            head_indices = self.test_head_indices\n            tail_indices = self.test_tail_indices\n\n        if len(head_indices) % self.batch_size == 0:\n            max_batch_num = len(head_indices) // self.batch_size\n        else:\n            max_batch_num = len(head_indices) // self.batch_size + 1\n        for batch_num in range(max_batch_num):\n            if (batch_num + 1) * self.batch_size <= len(head_indices):\n                head_batch = head_indices[batch_num * self.batch_size: (batch_num + 1) * self.batch_size]\n                tail_batch = tail_indices[batch_num * self.batch_size: (batch_num + 1) * self.batch_size]\n            else:\n                head_batch = head_indices[batch_num * self.batch_size:]\n                tail_batch = tail_indices[batch_num * self.batch_size:]\n\n            head_batch_indices = torch.LongTensor([indice[\'triple\'] for indice in head_batch])\n            head_batch_indices = head_batch_indices.to(self.device)\n            rel_ids = head_batch_indices[:, 1]\n            pred, attention = model.forward(head_batch_indices)\n            pred = pred[index]\n            label = [np.int32(indice[\'label\']) for indice in head_batch]\n            y = np.zeros((len(head_batch), len(self.entity2id)), dtype=np.float32)\n            for idx in range(len(label)):\n                for l in label[idx]:\n                    y[idx][l] = 1.0\n            y = torch.FloatTensor(y).to(self.device)\n            target = head_batch_indices[:, 2]\n            b_range = torch.arange(pred.shape[0], device=self.device)\n            target_pred = pred[b_range, target]\n            pred = torch.where(y.byte(), torch.zeros_like(pred), pred)\n            pred[b_range, target] = target_pred\n            pred = pred.cpu().numpy()\n            target = target.cpu().numpy()\n            for i in range(pred.shape[0]):\n                scores = pred[i]\n                tar = target[i]\n                tar_scr = scores[tar]\n                scores = np.delete(scores, tar)\n                rand = np.random.randint(scores.shape[0])\n                scores = np.insert(scores, rand, tar_scr)\n                sorted_indices = np.argsort(-scores, kind=\'stable\')\n                # higher is better\n                ranks_head.append(np.where(sorted_indices == rand)[0][0]+1)\n                reciprocal_ranks_head.append(1.0 / ranks_head[-1])\n                rel_pred_dict[rel_ids[i].item()].append(ranks_head[-1])\n\n            tail_batch_indices = torch.LongTensor([indice[\'triple\'] for indice in tail_batch])\n            tail_batch_indices = tail_batch_indices.to(self.device)\n            rel_ids = tail_batch_indices[:, 1]\n            pred, attention = model.forward(tail_batch_indices)\n            pred = pred[index]\n            label = [np.int32(indice[\'label\']) for indice in tail_batch]\n            y = np.zeros((len(tail_batch), len(self.entity2id)), dtype=np.float32)\n            for idx in range(len(label)):\n                for l in label[idx]:\n                    y[idx][l] = 1.0\n            y = torch.FloatTensor(y).to(self.device)\n            target = tail_batch_indices[:, 2]\n            b_range = torch.arange(pred.shape[0], device=self.device)\n            target_pred = pred[b_range, target]\n            pred = torch.where(y.byte(), torch.zeros_like(pred), pred)\n            pred[b_range, target] = target_pred\n            pred = pred.cpu().numpy()\n            target = target.cpu().numpy()\n            for i in range(pred.shape[0]):\n                scores = pred[i]\n                tar = target[i]\n                tar_scr = scores[tar]\n                scores = np.delete(scores, tar)\n                rand = np.random.randint(scores.shape[0])\n                scores = np.insert(scores, rand, tar_scr)\n                sorted_indices = np.argsort(-scores, kind=\'stable\')\n                ranks_tail.append(np.where(sorted_indices == rand)[0][0] + 1)\n                reciprocal_ranks_tail.append(1.0 / ranks_tail[-1])\n                rel_pred_dict[rel_ids[i].item()].append(ranks_head[-1])\n\n        for i in range(len(ranks_head)):\n            if ranks_head[i] <= 100:\n                hits_at_100_head += 1\n            if ranks_head[i] <= 10:\n                hits_at_10_head += 1\n            if ranks_head[i] <= 3:\n                hits_at_3_head += 1\n            if ranks_head[i] == 1:\n                hits_at_1_head += 1\n\n        for i in range(len(ranks_tail)):\n            if ranks_tail[i] <= 100:\n                hits_at_100_tail += 1\n            if ranks_tail[i] <= 10:\n                hits_at_10_tail += 1\n            if ranks_tail[i] <= 3:\n                hits_at_3_tail += 1\n            if ranks_tail[i] == 1:\n                hits_at_1_tail += 1\n\n        assert len(ranks_head) == len(reciprocal_ranks_head)\n        assert len(ranks_tail) == len(reciprocal_ranks_tail)\n\n        hits_100_head = hits_at_100_head / len(ranks_head)\n        hits_10_head = hits_at_10_head / len(ranks_head)\n        hits_3_head = hits_at_3_head / len(ranks_head)\n        hits_1_head = hits_at_1_head / len(ranks_head)\n        mean_rank_head = sum(ranks_head) / len(ranks_head)\n        mean_reciprocal_rank_head = sum(reciprocal_ranks_head) / len(reciprocal_ranks_head)\n\n        hits_100_tail = hits_at_100_tail / len(ranks_tail)\n        hits_10_tail = hits_at_10_tail / len(ranks_tail)\n        hits_3_tail = hits_at_3_tail / len(ranks_tail)\n        hits_1_tail = hits_at_1_tail / len(ranks_tail)\n        mean_rank_tail = sum(ranks_tail) / len(ranks_tail)\n        mean_reciprocal_rank_tail = sum(reciprocal_ranks_tail) / len(reciprocal_ranks_tail)\n\n        hits_100 = (hits_at_100_head / len(ranks_head) + hits_at_100_tail / len(ranks_tail)) / 2\n        hits_10 = (hits_at_10_head / len(ranks_head) + hits_at_10_tail / len(ranks_tail)) / 2\n        hits_3 = (hits_at_3_head / len(ranks_head) + hits_at_3_tail / len(ranks_tail)) / 2\n        hits_1 = (hits_at_1_head / len(ranks_head) + hits_at_1_tail / len(ranks_tail)) / 2\n        mean_rank = (sum(ranks_head) / len(ranks_head) + sum(ranks_tail) / len(ranks_tail)) / 2\n        mean_reciprocal_rank = (sum(reciprocal_ranks_head) / len(reciprocal_ranks_head) + sum(\n            reciprocal_ranks_tail) / len(reciprocal_ranks_tail)) / 2\n\n        metrics = {\n            "Hits@100": hits_100,\n            "Hits@10": hits_10,\n            "Hits@3": hits_3,\n            "Hits@1": hits_1,\n            "MR": mean_rank,\n            "MRR": mean_reciprocal_rank\n        }\n\n        return metrics, rel_pred_dict\n\n\nclass ConvKBCorpus(Corpus):\n    def __init__(self, args, train_data, val_data, test_data, entity2id, relation2id):\n        super(ConvKBCorpus, self).__init__(args, train_data, val_data, test_data, entity2id, relation2id)\n        self.neg_num = args.neg_num\n        if len(self.train_triples) % self.batch_size == 0:\n            self.max_batch_num = len(self.train_triples) // self.batch_size\n        else:\n            self.max_batch_num = len(self.train_triples) // self.batch_size + 1\n\n        self.train_indices = np.array(self.train_triples).astype(np.int32)\n        self.train_values = np.array([[1]] * len(self.train_triples)).astype(np.float32)\n        self.val_indices = np.array(self.val_triples).astype(np.int32)\n        self.val_values = np.array([[1]] * len(self.val_triples)).astype(np.float32)\n        self.test_indices = np.array(self.test_triples).astype(np.int32)\n        self.test_values = np.array([[1]] * len(self.test_triples)).astype(np.float32)\n\n        self.unique_entities = [entity2id[i] for i in train_data[2]]\n        self.all_triples = {j: i for i, j in enumerate(self.train_triples + self.val_triples + self.test_triples)}\n\n        self.batch_indices = np.empty((self.batch_size * (self.neg_num + 1), 3)).astype(np.int32)\n        self.batch_values = np.empty((self.batch_size * (self.neg_num + 1), 1)).astype(np.float32)\n\n    def shuffle(self):\n        np.random.shuffle(self.train_indices)\n\n    def get_batch(self, batch_num):\n        if (batch_num + 1) * self.batch_size <= len(self.train_indices):\n            self.batch_indices = np.empty((self.batch_size * (self.neg_num + 1), 3)).astype(np.int32)\n            self.batch_values = np.empty((self.batch_size * (self.neg_num + 1), 1)).astype(np.float32)\n\n            indices = range(self.batch_size * batch_num, self.batch_size * (batch_num + 1))\n            last_index = self.batch_size\n\n        else:\n            last_batch_size = len(self.train_indices) - self.batch_size * batch_num\n            self.batch_indices = np.empty((last_batch_size * (self.neg_num + 1), 3)).astype(np.int32)\n            self.batch_values = np.empty((last_batch_size * (self.neg_num + 1), 1)).astype(np.float32)\n\n            indices = range(self.batch_size * batch_num, len(self.train_indices))\n            last_index = last_batch_size\n\n        self.batch_indices[:last_index, :] = self.train_indices[indices, :]\n        self.batch_values[:last_index, :] = self.train_values[indices, :]\n        random_entities = np.random.randint(0, len(self.entity2id), last_index * self.neg_num)\n        self.batch_indices[last_index: (last_index * (self.neg_num + 1)), :] = np.tile(\n            self.batch_indices[:last_index, :], (self.neg_num, 1))\n        self.batch_values[last_index: (last_index * (self.neg_num + 1)), :] = np.tile(\n            self.batch_values[:last_index, :], (self.neg_num, 1))\n\n        for i in range(last_index):\n            for j in range(self.neg_num // 2):\n                current_index = i * (self.neg_num // 2) + j\n\n                while(random_entities[current_index], self.batch_indices[last_index + current_index, 1],\n                      self.batch_indices[last_index + current_index, 2]) in self.all_triples.keys():\n                    random_entities[current_index] = np.random.randint(0, len(self.entity2id))\n\n                self.batch_indices[last_index + current_index, 0] = random_entities[current_index]\n                self.batch_values[last_index + current_index, :] = [-1]\n            for j in range(self.neg_num // 2):\n                current_index = last_index * (self.neg_num // 2) + i * (self.neg_num // 2) + j\n\n                while (self.batch_indices[last_index + current_index, 0], self.batch_indices[last_index + current_index, 1],\n                       random_entities[current_index]) in self.all_triples.keys():\n                    random_entities[current_index] = np.random.randint(0, len(self.entity2id))\n\n                self.batch_indices[last_index + current_index, 2] = random_entities[current_index]\n                self.batch_values[last_index + current_index, :] = [-1]\n\n        return self.batch_indices, self.batch_values\n\n    def get_validation_pred(self, model, split=\'test\'):\n        ranks_head, ranks_tail = [], []\n        reciprocal_ranks_head, reciprocal_ranks_tail = [], []\n        hits_at_100_head, hits_at_100_tail = 0, 0\n        hits_at_10_head, hits_at_10_tail = 0, 0\n        hits_at_3_head, hits_at_3_tail = 0, 0\n        hits_at_1_head, hits_at_1_tail = 0, 0\n        entity_list = [i for i in self.entity2id.values()]\n        if split == \'val\':\n            split_triples = np.array(self.val_triples).astype(np.int32)\n        elif split == \'test\':\n            split_triples = np.array(self.test_triples).astype(np.int32)\n\n        for i in range(split_triples.shape[0]):\n            if split_triples[i, 0] not in self.unique_entities or split_triples[i, 2] not in self.unique_entities:\n                continue\n            x_head = np.tile(split_triples[i, :], (len(self.entity2id), 1))\n            x_tail = np.tile(split_triples[i, :], (len(self.entity2id), 1))\n            x_head[:, 0] = entity_list\n            x_tail[:, 2] = entity_list\n\n            last_index_head, last_index_tail = [], []\n            for idx in range(len(x_head)):\n                head = (x_head[idx][0], x_head[idx][1], x_head[idx][2])\n                if head in self.all_triples.keys():\n                    last_index_head.append(idx)\n\n                tail = (x_tail[idx][0], x_tail[idx][1], x_tail[idx][2])\n                if tail in self.all_triples.keys():\n                    last_index_tail.append(idx)\n\n            x_head = np.delete(x_head, last_index_head, axis=0)\n            x_tail = np.delete(x_tail, last_index_tail, axis=0)\n            rand_head = np.random.randint(x_head.shape[0])\n            rand_tail = np.random.randint(x_tail.shape[0])\n            x_head = np.insert(x_head, rand_head, split_triples[i], axis=0)\n            x_tail = np.insert(x_tail, rand_tail, split_triples[i], axis=0)\n            x_head = torch.LongTensor(x_head).to(self.device)\n            x_tail = torch.LongTensor(x_tail).to(self.device)\n            #scores_head = model.forward(x_head)\n            scores_head = model.predict(x_head)\n            sorted_scores_head, sorted_triples_head = torch.sort(scores_head.view(-1), dim=-1, descending=True)\n            ranks_head.append(np.where(sorted_triples_head.cpu().numpy() == rand_head)[0][0]+1)\n            reciprocal_ranks_head.append(1.0 / ranks_head[-1])\n            #scores_tail = model.forward(x_tail)\n            scores_tail = model.predict(x_tail)\n            sorted_scores_tail, sorted_triples_tail = torch.sort(scores_tail.view(-1), dim=-1, descending=True)\n            ranks_tail.append(np.where(sorted_triples_tail.cpu().numpy() == rand_tail)[0][0]+1)\n            reciprocal_ranks_tail.append(1.0 / ranks_tail[-1])\n\n        for i in range(len(ranks_head)):\n            if ranks_head[i] <= 100:\n                hits_at_100_head += 1\n            if ranks_head[i] <= 10:\n                hits_at_10_head += 1\n            if ranks_head[i] <= 3:\n                hits_at_3_head += 1\n            if ranks_head[i] == 1:\n                hits_at_1_head += 1\n\n        for i in range(len(ranks_tail)):\n            if ranks_tail[i] <= 100:\n                hits_at_100_tail += 1\n            if ranks_tail[i] <= 10:\n                hits_at_10_tail += 1\n            if ranks_tail[i] <= 3:\n                hits_at_3_tail += 1\n            if ranks_tail[i] == 1:\n                hits_at_1_tail += 1\n\n        assert len(ranks_head) == len(reciprocal_ranks_head)\n        assert len(ranks_tail) == len(reciprocal_ranks_tail)\n\n        hits_100_head = hits_at_100_head / len(ranks_head)\n        hits_10_head = hits_at_10_head / len(ranks_head)\n        hits_3_head = hits_at_3_head / len(ranks_head)\n        hits_1_head = hits_at_1_head / len(ranks_head)\n        mean_rank_head = sum(ranks_head) / len(ranks_head)\n        mean_reciprocal_rank_head = sum(reciprocal_ranks_head) / len(reciprocal_ranks_head)\n\n        hits_100_tail = hits_at_100_tail / len(ranks_tail)\n        hits_10_tail = hits_at_10_tail / len(ranks_tail)\n        hits_3_tail = hits_at_3_tail / len(ranks_tail)\n        hits_1_tail = hits_at_1_tail / len(ranks_tail)\n        mean_rank_tail = sum(ranks_tail) / len(ranks_tail)\n        mean_reciprocal_rank_tail = sum(reciprocal_ranks_tail) / len(reciprocal_ranks_tail)\n\n        hits_100 = (hits_at_100_head / len(ranks_head) + hits_at_100_tail / len(ranks_tail)) / 2\n        hits_10 = (hits_at_10_head / len(ranks_head) + hits_at_10_tail / len(ranks_tail)) / 2\n        hits_3 = (hits_at_3_head / len(ranks_head) + hits_at_3_tail / len(ranks_tail)) / 2\n        hits_1 = (hits_at_1_head / len(ranks_head) + hits_at_1_tail / len(ranks_tail)) / 2\n        mean_rank = (sum(ranks_head) / len(ranks_head) + sum(ranks_tail) / len(ranks_tail)) / 2\n        mean_reciprocal_rank = (sum(reciprocal_ranks_head) / len(reciprocal_ranks_head) + sum(reciprocal_ranks_tail) / len(reciprocal_ranks_tail)) / 2\n\n        metrics = {\n            "Hits@100_head": hits_100_head,\n            "Hits@10_head": hits_10_head,\n            "Hits@3_head": hits_3_head,\n            "Hits@1_head": hits_1_head,\n            "Mean Rank_head": mean_rank_head,\n            "Mean Reciprocal Rank_head": mean_reciprocal_rank_head,\n            "Hits@100_tail": hits_100_tail, "Hits@10_tail": hits_10_tail, "Hits@3_tail": hits_3_tail, "Hits@1_tail": hits_1_tail,\n            "Mean Rank_tail": mean_rank_tail, "Mean Reciprocal Rank_tail": mean_reciprocal_rank_tail,\n            "Hits@100": hits_100, "Hits@10": hits_10, "Hits@3": hits_3, "Hits@1": hits_1,\n            "Mean Rank": mean_rank, "Mean Reciprocal Rank": mean_reciprocal_rank}\n\n        return metrics\n\n\n', 'utils/data_util.py': "import h5py\nimport pickle\nimport torch\nimport numpy as np\n\n\n# Get item2id and write into txt\ndef write_index_dict(datasets):\n    path = 'datasets/'+datasets+'/'\n    entities = set()\n    relations = set()\n    with open(path+datasets+'_EntityTriples.txt', 'r') as f:\n        for line in f:\n            instance = line.strip().split(' ')\n            entities.add(instance[0])\n            relations.add(instance[1])\n            entities.add(instance[2])\n\n    with open(path+'entity2id.txt', 'w') as f:\n        for index, entity in enumerate(entities):\n            f.write(entity+' '+str(index)+'\\n')\n\n    with open(path+'relation2id.txt', 'w') as f:\n        for index, relation in enumerate(relations):\n            f.write(relation+' '+str(index)+'\\n')\n\n\n# Bind img features with id\ndef write_img_vec(datasets):\n    path = 'datasets/'+datasets+'/'\n    entities = {}\n    with open(path + datasets + '_ImageIndex.txt', 'r') as f:\n        for line in f:\n            instance = line.strip().split('\\t')\n            entities[instance[0]] = instance[1]\n\n    img_features = []\n    with open(path + 'entity2id.txt', 'r') as f:\n        with h5py.File(path + datasets + '_ImageData.h5', 'r') as img:\n            img_all = np.array([feats for feats in img.values()])\n            img_mean = np.mean(img_all.reshape(-1, img_all.shape[2]), 0)\n            for line in f:\n                instance = line.strip().split(' ')\n                entity = instance[0]\n                if entity in entities.keys():\n                    img_features.append(np.array(img[entities[entity]]).flatten())\n                else:\n                    img_features.append(img_mean)\n\n    img_features = np.array(img_features)\n    pickle.dump(img_features, open(path + 'img_features.pkl', 'wb'))\n\n\ndef data_preprocess(datasets):\n    write_index_dict(datasets)\n    write_img_vec(datasets)\n    dataset_split(datasets)\n\n\ndef read_entity_from_id(path):\n    entity2id = {}\n    with open(path + 'entity2id.txt', 'r') as f:\n        for line in f:\n            instance = line.strip().split()\n            entity2id[instance[0]] = int(instance[1])\n\n    return entity2id\n\n\ndef read_relation_from_id(path):\n    relation2id = {}\n    with open(path + 'relation2id.txt', 'r') as f:\n        for line in f:\n            instance = line.strip().split()\n            relation2id[instance[0]] = int(instance[1])\n\n    return relation2id\n\n\n# Calculate adjacency matrix\ndef get_adj(path, split):\n    entity2id = read_entity_from_id(path)\n    relation2id = read_relation_from_id(path)\n    triples = []\n    rows, cols, data = [], [], []\n    unique_entities = set()\n    with open(path+split+'.txt', 'r') as f:\n        for line in f:\n            instance = line.strip().split(' ')\n            e1, r, e2 = instance[0], instance[1], instance[2]\n            unique_entities.add(e1)\n            unique_entities.add(e2)\n            triples.append((entity2id[e1], relation2id[r], entity2id[e2]))\n            rows.append(entity2id[e2])\n            cols.append(entity2id[e1])\n            data.append(relation2id[r])\n\n    return triples, (rows, cols, data), unique_entities\n\n\n# Load data triples and adjacency matrix\ndef load_data(datasets):\n    path = 'datasets/'+datasets+'/'\n    train_triples, train_adj, train_unique_entities = get_adj(path, 'train')\n    val_triples, val_adj, val_unique_entities = get_adj(path, 'valid')\n    test_triples, test_adj, test_unique_entities = get_adj(path, 'test')\n    entity2id = read_entity_from_id(path)\n    relation2id = read_relation_from_id(path)\n    img_features = torch.load(open(path+'img_features.pth', 'rb'))\n    text_features = torch.load(open(path+'text_features.pth', 'rb'))\n\n    return entity2id, relation2id, img_features, text_features, \\\n           (train_triples, train_adj, train_unique_entities), \\\n           (val_triples, val_adj, val_unique_entities), \\\n           (test_triples, test_adj, test_unique_entities)\n\n\ndef load_more_data(datasets, numeric=True):\n    path = 'datasets/'+datasets+'/'\n    train_triples, train_adj, train_unique_entities = get_adj(path, 'train')\n    val_triples, val_adj, val_unique_entities = get_adj(path, 'valid')\n    test_triples, test_adj, test_unique_entities = get_adj(path, 'test')\n    entity2id = read_entity_from_id(path)\n    relation2id = read_relation_from_id(path)\n    img_features = torch.load(open(path+'img_features.pth', 'rb'))\n    text_features = torch.load(open(path+'text_features.pth', 'rb'))\n    if numeric:\n        num_features = torch.load(open(path+'numeric_features.pth', 'rb'))\n    else:\n        num_features = None\n    print(num_features)\n    return entity2id, relation2id, img_features, text_features, num_features \\\n           (train_triples, train_adj, train_unique_entities), \\\n           (val_triples, val_adj, val_unique_entities), \\\n           (test_triples, test_adj, test_unique_entities)\n\n\n# Split data into train, val and test\ndef dataset_split(datasets):\n    path = 'datasets/' + datasets + '/'\n    with open(path+datasets+'_EntityTriples.txt', 'r') as f:\n        triples = f.readlines()\n\n    np.random.shuffle(triples)\n    nb_val = round(0.05 * len(triples))\n    nb_test = round(0.05 * len(triples))\n    val_triples, test_triples, train_triples = triples[:nb_val], triples[nb_val: nb_val+nb_test], triples[nb_val+nb_test:]\n\n    with open(path+'train.txt', 'w') as f:\n        f.writelines(train_triples)\n    with open(path+'valid.txt', 'w') as f:\n        f.writelines(val_triples)\n    with open(path+'test.txt', 'w') as f:\n        f.writelines(test_triples)\n\n\ndef data_loader(datasets):\n    path = 'datasets/'+datasets\n    with open(path+'/'+datasets+'_EntityTriples.txt', 'r') as f:\n        for line in f:\n            print(line)\n", 'layers/layer.py': 'import numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nCUDA = torch.cuda.is_available()\n\n\nclass ConvKBLayer(nn.Module):\n    def __init__(self, input_dim, input_seq_len, in_channels, out_channels, drop_prob, alpha_leaky):\n        super(ConvKBLayer, self).__init__()\n\n        self.conv_layer = nn.Conv2d(in_channels, out_channels, (1, input_seq_len))\n        self.dropout = nn.Dropout(drop_prob)\n        self.non_linearity = nn.ReLU()\n        self.fc_layer = nn.Linear(input_dim * out_channels, 1)\n\n        nn.init.xavier_uniform_(self.fc_layer.weight, gain=1.414)\n        nn.init.xavier_uniform_(self.conv_layer.weight, gain=1.414)\n\n    def forward(self, conv_input):\n        batch_size, length, dim = conv_input.size()\n        conv_input = conv_input.transpose(1, 2)\n        conv_input = conv_input.unsqueeze(1)\n        out_conv = self.dropout(self.non_linearity(self.conv_layer(conv_input)))\n        input_fc = out_conv.squeeze(-1).view(batch_size, -1)\n        output = self.fc_layer(input_fc)\n        return output\n\n\nclass MutanLayer(nn.Module):\n    def __init__(self, dim, multi):\n        super(MutanLayer, self).__init__()\n\n        self.dim = dim\n        self.multi = multi\n\n        modal1 = []\n        for i in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(dim, dim)\n            modal1.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal1_layers = nn.ModuleList(modal1)\n\n        modal2 = []\n        for i in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(dim, dim)\n            modal2.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal2_layers = nn.ModuleList(modal2)\n\n        modal3 = []\n        for i in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(dim, dim)\n            modal3.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal3_layers = nn.ModuleList(modal3)\n\n    def forward(self, modal1_emb, modal2_emb, modal3_emb):\n        bs = modal1_emb.size(0)\n        x_mm = []\n        for i in range(self.multi):\n            x_modal1 = self.modal1_layers[i](modal1_emb)\n            x_modal2 = self.modal2_layers[i](modal2_emb)\n            x_modal3 = self.modal3_layers[i](modal3_emb)\n            x_mm.append(torch.mul(torch.mul(x_modal1, x_modal2), x_modal3))\n        x_mm = torch.stack(x_mm, dim=1)\n        x_mm = x_mm.sum(1).view(bs, self.dim)\n        x_mm = torch.relu(x_mm)\n        return x_mm\n\n\nclass TuckERLayer(nn.Module):\n    def __init__(self, dim, r_dim):\n        super(TuckERLayer, self).__init__()\n        \n        self.W = nn.Parameter(torch.rand(r_dim, dim, dim))\n        nn.init.xavier_uniform_(self.W.data)\n        self.bn0 = nn.BatchNorm1d(dim)\n        self.bn1 = nn.BatchNorm1d(dim)\n        self.input_drop = nn.Dropout(0.3)\n        self.hidden_drop = nn.Dropout(0.4)\n        self.out_drop = nn.Dropout(0.5)\n\n    def forward(self, e_embed, r_embed):\n        x = self.bn0(e_embed)\n        x = self.input_drop(x)\n        x = x.view(-1, 1, x.size(1))\n        \n        r = torch.mm(r_embed, self.W.view(r_embed.size(1), -1))\n        r = r.view(-1, x.size(2), x.size(2))\n        r = self.hidden_drop(r)\n       \n        x = torch.bmm(x, r)\n        x = x.view(-1, x.size(2))\n        x = self.bn1(x)\n        x = self.out_drop(x)\n        return x\n\n\nclass ConvELayer(nn.Module):\n    def __init__(self, dim, out_channels, kernel_size, k_h, k_w):\n        super(ConvELayer, self).__init__()\n\n        self.input_drop = nn.Dropout(0.2)\n        self.conv_drop = nn.Dropout2d(0.2)\n        self.hidden_drop = nn.Dropout(0.2)\n        self.bn0 = nn.BatchNorm2d(1)\n        self.bn1 = nn.BatchNorm2d(out_channels)\n        self.bn2 = nn.BatchNorm1d(dim)\n\n        self.conv = torch.nn.Conv2d(1, out_channels=out_channels, kernel_size=(kernel_size, kernel_size),\n                                    stride=1, padding=0, bias=True)\n        assert k_h * k_w == dim\n        flat_sz_h = int(2*k_w) - kernel_size + 1\n        flat_sz_w = k_h - kernel_size + 1\n        self.flat_sz = flat_sz_h * flat_sz_w * out_channels\n        self.fc = nn.Linear(self.flat_sz, dim, bias=True)\n\n    def forward(self, conv_input):\n        x = self.bn0(conv_input)\n        x = self.input_drop(x)\n        x = self.conv(x)\n        x = self.bn1(x)\n        x = F.relu(x)\n        x = self.conv_drop(x)\n        x = x.view(-1, self.flat_sz)\n        x = self.fc(x)\n        x = self.hidden_drop(x)\n        x = self.bn2(x)\n        x = F.relu(x)\n        return x\n\n\nclass SpecialSpmmFunctionFinal(torch.autograd.Function):\n    """Special function for only sparse region backpropataion layer."""\n    @staticmethod\n    def forward(ctx, edge, edge_w, N, E, out_features):\n        a = torch.sparse_coo_tensor(\n            edge, edge_w, torch.Size([N, N, out_features]))\n        b = torch.sparse.sum(a, dim=1)\n        ctx.N = b.shape[0]\n        ctx.outfeat = b.shape[1]\n        ctx.E = E\n        ctx.indices = a._indices()[0, :]\n\n        return b.to_dense()\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        grad_values = None\n        if ctx.needs_input_grad[1]:\n            edge_sources = ctx.indices\n            if(CUDA):\n                edge_sources = edge_sources.to(\'cuda:0\')\n            grad_values = grad_output[edge_sources]\n        return None, grad_values, None, None, None\n\n\nclass SpecialSpmmFinal(nn.Module):\n    def forward(self, edge, edge_w, N, E, out_features):\n        return SpecialSpmmFunctionFinal.apply(edge, edge_w, N, E, out_features)\n\n\nclass SpGraphAttentionLayer(nn.Module):\n    """\n    Sparse version GAT layer, similar to https://arxiv.org/abs/1710.10903\n    """\n\n    def __init__(self, num_nodes, in_features, out_features, dropout, alpha, concat=True):\n        super(SpGraphAttentionLayer, self).__init__()\n        self.in_features = in_features\n        self.out_features = out_features\n        self.num_nodes = num_nodes\n        self.alpha = alpha\n        self.concat = concat\n\n        self.W = nn.Parameter(torch.zeros(size=(out_features, 2 * in_features)))\n        nn.init.xavier_normal_(self.W.data, gain=1.414)\n        self.a = nn.Parameter(torch.zeros(size=(1, out_features)))\n        nn.init.xavier_normal_(self.a.data, gain=1.414)\n\n        self.dropout = nn.Dropout(dropout)\n        self.leakyrelu = nn.LeakyReLU(self.alpha)\n        self.special_spmm_final = SpecialSpmmFinal()\n\n    def forward(self, input, edge):\n        N = input.size()[0]\n\n        edge_h = torch.cat((input[edge[0, :], :], input[edge[1, :], :]), dim=1).t()\n        # edge_h: (2*in_dim) x E\n\n        edge_m = self.W.mm(edge_h)\n        # edge_m: D * E\n\n        # to be checked later\n        edge_e = torch.exp(-self.leakyrelu(self.a.mm(edge_m).squeeze())).unsqueeze(1)\n        assert not torch.isnan(edge_e).any()\n        # edge_e: E\n\n        e_rowsum = self.special_spmm_final(edge, edge_e, N, edge_e.shape[0], 1)\n        e_rowsum[e_rowsum == 0.0] = 1e-12\n\n        e_rowsum = e_rowsum\n        # e_rowsum: N x 1\n        edge_e = edge_e.squeeze(1)\n\n        edge_e = self.dropout(edge_e)\n        # edge_e: E\n\n        edge_w = (edge_e * edge_m).t()\n        # edge_w: E * D\n\n        h_prime = self.special_spmm_final(edge, edge_w, N, edge_w.shape[0], self.out_features)\n        assert not torch.isnan(h_prime).any()\n        h_prime = h_prime.div(e_rowsum)\n        assert not torch.isnan(h_prime).any()\n        if self.concat:\n            # if this layer is not last layer,\n            return F.elu(h_prime)\n        else:\n            # if this layer is last layer,\n            return h_prime\n\n    def __repr__(self):\n        return self.__class__.__name__ + \' (\' + str(self.in_features) + \' -> \' + str(self.out_features) + \')\'\n\n\nclass SpGAT(nn.Module):\n    def __init__(self, num_nodes, nfeat, nhid, dropout, alpha, nheads):\n        """\n            Sparse version of GAT\n            nfeat -> Entity Input Embedding dimensions\n            nhid  -> Entity Output Embedding dimensions\n            num_nodes -> number of nodes in the Graph\n            nheads -> Used for Multihead attention\n        """\n        super(SpGAT, self).__init__()\n        self.dropout = dropout\n        self.dropout_layer = nn.Dropout(self.dropout)\n        self.attentions = [SpGraphAttentionLayer(num_nodes, nfeat,\n                                                 nhid,\n                                                 dropout=dropout,\n                                                 alpha=alpha,\n                                                 concat=True)\n                           for _ in range(nheads)]\n\n        for i, attention in enumerate(self.attentions):\n            self.add_module(\'attention_{}\'.format(i), attention)\n\n        # W matrix to convert h_input to h_output dimension\n        self.W = nn.Parameter(torch.zeros(size=(nfeat, nheads * nhid)))\n        nn.init.xavier_uniform_(self.W.data, gain=1.414)\n\n        self.out_att = SpGraphAttentionLayer(num_nodes, nheads * nhid,\n                                             nheads * nhid,\n                                             dropout=dropout,\n                                             alpha=alpha,\n                                             concat=False)\n\n    def forward(self, entity_embeddings, relation_embeddings, edge_list):\n        x = entity_embeddings\n        x = torch.cat([att(x, edge_list) for att in self.attentions], dim=1)\n        x = self.dropout_layer(x)\n        x_rel = relation_embeddings.mm(self.W)\n        x = F.elu(self.out_att(x, edge_list))\n        return x, x_rel\n\n\nclass ModalFusionLayerForThreeModal(nn.Module):\n    def __init__(self, in_dim, out_dim, multi, img_dim, txt_dim):\n        super(ModalFusionLayerForThreeModal, self).__init__()\n\n        self.in_dim = in_dim\n        self.out_dim = out_dim\n        self.multi = multi\n        self.img_dim = img_dim\n        self.text_dim = txt_dim\n\n        modal1 = []\n        for _ in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(in_dim, out_dim)\n            modal1.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal1_layers = nn.ModuleList(modal1)\n\n        modal2 = []\n        for _ in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(self.img_dim, out_dim)\n            modal2.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal2_layers = nn.ModuleList(modal2)\n\n        modal3 = []\n        for _ in range(self.multi):\n            do = nn.Dropout(p=0.2)\n            lin = nn.Linear(self.text_dim, out_dim)\n            modal3.append(nn.Sequential(do, lin, nn.ReLU()))\n        self.modal3_layers = nn.ModuleList(modal3)\n\n        self.ent_attn = nn.Linear(self.out_dim, 1, bias=False)\n        self.ent_attn.requires_grad_(True)\n\n    def forward(self, modal1_emb, modal2_emb, modal3_emb):\n        batch_size = modal1_emb.size(0)\n        x_mm = []\n        for i in range(self.multi):\n            x_modal1 = self.modal1_layers[i](modal1_emb)\n            x_modal2 = self.modal2_layers[i](modal2_emb)\n            x_modal3 = self.modal3_layers[i](modal3_emb)\n            x_stack = torch.stack((x_modal1, x_modal2, x_modal3), dim=1)\n            attention_scores = self.ent_attn(x_stack).squeeze(-1)\n            attention_weights = torch.softmax(attention_scores, dim=-1)\n            context_vectors = torch.sum(attention_weights.unsqueeze(-1) * x_stack, dim=1)\n            x_mm.append(context_vectors)\n        x_mm = torch.stack(x_mm, dim=1)\n        x_mm = x_mm.sum(1).view(batch_size, self.out_dim)\n        # x_mm = torch.relu(x_mm)\n        return x_mm', 'ablations/__init__.py': '"""Configuration and execution helpers for MoMoK ablation experiments."""\n\nfrom .variants import VariantConfig, VariantConfigError, resolve_variant\n\n__all__ = ["VariantConfig", "VariantConfigError", "resolve_variant"]\n', 'ablations/variants.py': '"""Explicit operational definitions for MoMoK ablation variants.\n\nThis module is intentionally framework-independent.  It is the contract\nbetween experiment configuration, model/loss integration, checkpoint\nmetadata, and reporting.  It does not import PyTorch and performs no\ntraining.\n"""\n\nimport hashlib\nimport json\nfrom dataclasses import asdict, dataclass\nfrom typing import Tuple\n\n\nclass VariantConfigError(ValueError):\n    """Raised when an ablation configuration is invalid or ambiguous."""\n\n\nHEADS = ("structure", "image", "text", "joint")\nSINGLE_MODALITY_VARIANTS = {\n    "structure_only": ("structure",),\n    "image_only": ("image",),\n    "text_only": ("text",),\n}\nVARIANT_IDS = frozenset(\n    {\n        "full",\n        *SINGLE_MODALITY_VARIANTS,\n        "joint_only",\n        "no_relation_temperature",\n        "no_gating_noise",\n        "no_adaptive_fusion",\n        "no_joint_training_loss_only",\n        "no_joint_head",\n        "no_exid",\n        "mu_zero_proxy",\n        "k_sweep",\n        "mu_sweep",\n    }\n)\n\n\ndef config_fingerprint(config: dict) -> str:\n    """Hash canonical JSON so output/checkpoint namespaces are stable."""\n\n    canonical = json.dumps(config, sort_keys=True, separators=(",", ":"), default=str)\n    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()\n\n\n@dataclass(frozen=True)\nclass VariantConfig:\n    """Serializable variant behavior consumed by later runtime adapters."""\n\n    variant_id: str\n    active_heads: Tuple[str, ...]\n    use_relation_temperature: bool\n    use_gating_noise: bool\n    use_adaptive_fusion: bool\n    fusion_mode: str\n    train_joint_head: bool\n    infer_joint_head: bool\n    use_exid: bool\n    exid_mode: str\n    n_exp: int\n    mu: float\n\n    def to_dict(self) -> dict:\n        """Return JSON-compatible configuration metadata."""\n\n        result = asdict(self)\n        result["active_heads"] = list(self.active_heads)\n        return result\n\n\ndef resolve_variant(variant_id: str, *, n_exp: int, mu: float) -> VariantConfig:\n    """Resolve an explicit variant contract without executing model code.\n\n    ``k_sweep`` is the only variant that permits ``n_exp=1``.  In that\n    case ExID is disabled explicitly because the current CLUB estimator\n    requires two distinct experts.  This prevents silently reporting a\n    K=1 run as if it used the normal ExID objective.\n    """\n\n    if variant_id not in VARIANT_IDS:\n        raise VariantConfigError(f"unknown variant: {variant_id}")\n    if not isinstance(n_exp, int) or n_exp < 1:\n        raise VariantConfigError("n_exp must be a positive integer")\n\n    if variant_id != "k_sweep" and n_exp < 2:\n        raise VariantConfigError(\n            f"{variant_id} requires n_exp >= 2 because CLUB samples two experts"\n        )\n\n    if variant_id == "k_sweep" and n_exp == 1:\n        use_exid = False\n        exid_mode = "disabled_k1"\n        effective_mu = 0.0\n    elif variant_id == "no_exid":\n        if mu != 0.0:\n            raise VariantConfigError("no_exid requires mu=0")\n        use_exid = False\n        exid_mode = "disabled"\n        effective_mu = 0.0\n    elif variant_id == "mu_zero_proxy":\n        if mu != 0.0:\n            raise VariantConfigError("mu_zero_proxy requires mu=0")\n        use_exid = True\n        exid_mode = "mu_zero_proxy"\n        effective_mu = 0.0\n    else:\n        use_exid = True\n        exid_mode = "active"\n        effective_mu = float(mu)\n\n    active_heads = HEADS\n    train_joint_head = True\n    infer_joint_head = True\n    use_relation_temperature = True\n    use_gating_noise = True\n    use_adaptive_fusion = True\n    fusion_mode = "adaptive"\n\n    if variant_id in SINGLE_MODALITY_VARIANTS:\n        active_heads = SINGLE_MODALITY_VARIANTS[variant_id]\n        train_joint_head = False\n        infer_joint_head = False\n    elif variant_id == "joint_only":\n        active_heads = ("joint",)\n        train_joint_head = True\n        infer_joint_head = True\n    elif variant_id == "no_relation_temperature":\n        use_relation_temperature = False\n    elif variant_id == "no_gating_noise":\n        use_gating_noise = False\n    elif variant_id == "no_adaptive_fusion":\n        use_adaptive_fusion = False\n        fusion_mode = "equal"\n    elif variant_id == "no_joint_training_loss_only":\n        train_joint_head = False\n    elif variant_id == "no_joint_head":\n        train_joint_head = False\n        infer_joint_head = False\n        active_heads = ("structure", "image", "text")\n\n    return VariantConfig(\n        variant_id=variant_id,\n        active_heads=active_heads,\n        use_relation_temperature=use_relation_temperature,\n        use_gating_noise=use_gating_noise,\n        use_adaptive_fusion=use_adaptive_fusion,\n        fusion_mode=fusion_mode,\n        train_joint_head=train_joint_head,\n        infer_joint_head=infer_joint_head,\n        use_exid=use_exid,\n        exid_mode=exid_mode,\n        n_exp=n_exp,\n        mu=effective_mu,\n    )\n', 'ablations/runtime_contract.py': '"""Pure runtime decisions shared by training, inference, and tests."""\n\nfrom .variants import VariantConfig\n\n\ndef training_heads(config: VariantConfig) -> tuple[str, ...]:\n    """Return prediction heads that contribute supervised training loss."""\n\n    heads = list(config.active_heads)\n    if not config.train_joint_head and "joint" in heads:\n        heads.remove("joint")\n    if config.variant_id == "no_joint_training_loss_only":\n        heads = [head for head in heads if head != "joint"]\n    return tuple(heads)\n\n\ndef inference_heads(config: VariantConfig) -> tuple[str, ...]:\n    """Return prediction heads used for the variant\'s reported inference."""\n\n    heads = list(config.active_heads)\n    if not config.infer_joint_head and "joint" in heads:\n        heads.remove("joint")\n    return tuple(heads)\n\n\ndef estimator_should_run(config: VariantConfig) -> bool:\n    """Whether the CLUB estimator and its optimizer are part of the run."""\n\n    return config.use_exid\n', 'ablations/checkpoint_contract.py': '"""Framework-independent checkpoint schema for ablation runs."""\n\nfrom typing import Iterable\n\n\nBASE_STATE_KEYS = (\n    "format_version",\n    "completed_epoch",\n    "model_state_dict",\n    "optimizer_state_dict",\n    "lr_scheduler_state_dict",\n    "best_validation_metrics",\n    "python_random_state",\n    "numpy_rng_state",\n    "torch_rng_state",\n    "torch_cuda_rng_state_all",\n    "corpus_train_indices",\n    "config",\n    "config_hash",\n    "variant_id",\n    "dataset",\n    "source_commit",\n)\nESTIMATOR_STATE_KEYS = ("estimator_state_dict", "optimizer_mi_state_dict")\n\n\ndef required_state_keys(*, use_exid: bool) -> tuple[str, ...]:\n    """Return required checkpoint keys for a variant\'s estimator mode."""\n\n    keys = list(BASE_STATE_KEYS)\n    if use_exid:\n        keys.extend(ESTIMATOR_STATE_KEYS)\n    return tuple(keys)\n\n\ndef build_checkpoint_metadata(\n    *,\n    variant_id: str,\n    config_hash: str,\n    dataset: str,\n    source_commit: str,\n    completed_epoch: int,\n    n_exp: int,\n) -> dict:\n    """Build the non-tensor metadata shared by save and resume validation."""\n\n    if completed_epoch < 0:\n        raise ValueError("completed_epoch must be non-negative")\n    if n_exp < 1:\n        raise ValueError("n_exp must be positive")\n    return {\n        "format_version": 1,\n        "completed_epoch": int(completed_epoch),\n        "variant_id": variant_id,\n        "config_hash": config_hash,\n        "dataset": dataset,\n        "source_commit": source_commit,\n        "n_exp": int(n_exp),\n    }\n\n\ndef missing_required_keys(payload: dict, *, use_exid: bool) -> tuple[str, ...]:\n    """Return missing schema keys without loading tensor values."""\n\n    required = required_state_keys(use_exid=use_exid)\n    return tuple(key for key in required if key not in payload)\n', 'ablations/checkpoints.py': '"""Atomic checkpoint and schema-validation helpers for ablation runs."""\n\nimport hashlib\nimport os\nfrom pathlib import Path\nfrom typing import Callable\n\nfrom .checkpoint_contract import missing_required_keys\n\n\ndef atomic_save(payload, path: Path, serializer: Callable | None = None) -> str:\n    """Atomically save bytes or a serialized payload and write its checksum."""\n\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if serializer is None:\n        if not isinstance(payload, (bytes, bytearray)):\n            raise TypeError("serializer is required for non-byte payloads")\n        data = bytes(payload)\n    else:\n        data = serializer(payload)\n        if not isinstance(data, (bytes, bytearray)):\n            raise TypeError("serializer must return bytes")\n        data = bytes(data)\n\n    temporary = path.with_name(path.name + ".tmp")\n    with temporary.open("wb") as handle:\n        handle.write(data)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, path)\n\n    digest = hashlib.sha256(data).hexdigest()\n    checksum_path = path.with_name(path.name + ".sha256")\n    checksum_path.write_text(f"{digest}  {path.name}\\n")\n    return digest\n\n\ndef validate_payload(payload: dict, *, expected_config: dict, use_exid: bool) -> dict:\n    """Validate schema and exact expected configuration before resume."""\n\n    missing = missing_required_keys(payload, use_exid=use_exid)\n    if missing:\n        raise ValueError(f"checkpoint missing required keys: {list(missing)}")\n    saved_config = payload.get("config", {})\n    mismatches = {\n        key: (saved_config.get(key), value)\n        for key, value in expected_config.items()\n        if saved_config.get(key) != value\n    }\n    if mismatches:\n        raise ValueError(f"checkpoint configuration mismatch: {mismatches}")\n    return payload\n', 'ablations/torch_checkpoint.py': '"""Torch-backed full-state checkpoint helpers.\n\nPyTorch is imported lazily so configuration and schema tests remain runnable\non the local CPU-only host.\n"""\n\nimport copy\nimport io\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .checkpoints import atomic_save, validate_payload\nfrom .variants import config_fingerprint\n\n\ndef checkpoint_config(args) -> dict:\n    variant = getattr(args, "variant_config", None)\n    return {\n        "dataset": args.dataset,\n        "variant": args.variant,\n        "seed": args.seed,\n        "lr": args.lr,\n        "mu": args.mu,\n        "dim": args.dim,\n        "r_dim": args.r_dim,\n        "batch_size": args.batch_size,\n        "n_exp": args.n_exp,\n        "epochs": args.epochs,\n        "eval_freq": args.eval_freq,\n        "metric_policy": getattr(args, "metric_policy", "legacy"),\n        "variant_config": variant.to_dict() if variant is not None else {},\n    }\n\n\ndef build_payload(\n    args,\n    model,\n    estimator,\n    optimizer,\n    optimizer_mi,\n    lr_scheduler,\n    best_validation_metrics,\n    corpus,\n    completed_epoch: int,\n    final_epoch_metrics=None,\n) -> dict:\n    import torch\n\n    config = checkpoint_config(args)\n    payload = {\n        "format_version": 1,\n        "completed_epoch": int(completed_epoch),\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "lr_scheduler_state_dict": lr_scheduler.state_dict(),\n        "best_validation_metrics": copy.deepcopy(best_validation_metrics),\n        "final_epoch_metrics": copy.deepcopy(final_epoch_metrics),\n        "python_random_state": random.getstate(),\n        "numpy_rng_state": np.random.get_state(),\n        "torch_rng_state": torch.get_rng_state(),\n        "torch_cuda_rng_state_all": torch.cuda.get_rng_state_all()\n        if torch.cuda.is_available()\n        else [],\n        "corpus_train_indices": copy.deepcopy(corpus.train_indices),\n        "config": config,\n        "config_hash": config_fingerprint(config),\n        "variant_id": args.variant,\n        "dataset": args.dataset,\n        "source_commit": os.environ.get("MOMOK_SOURCE_COMMIT", ""),\n        "n_exp": args.n_exp,\n    }\n    if estimator is not None and optimizer_mi is not None:\n        payload["estimator_state_dict"] = estimator.state_dict()\n        payload["optimizer_mi_state_dict"] = optimizer_mi.state_dict()\n    return payload\n\n\ndef save_checkpoint(path: Path, payload: dict) -> str:\n    import torch\n\n    def serialize(value):\n        buffer = io.BytesIO()\n        torch.save(value, buffer)\n        return buffer.getvalue()\n\n    return atomic_save(payload, Path(path), serializer=serialize)\n\n\ndef _move_optimizer_state(optimizer, device) -> None:\n    if optimizer is None:\n        return\n    for state in optimizer.state.values():\n        for key, value in state.items():\n            if hasattr(value, "to"):\n                state[key] = value.to(device)\n\n\ndef restore_checkpoint(\n    path: Path,\n    *,\n    args,\n    model,\n    estimator,\n    optimizer,\n    optimizer_mi,\n    lr_scheduler,\n    corpus,\n) -> dict:\n    import torch\n\n    payload = torch.load(path, map_location="cpu", weights_only=False)\n    config = checkpoint_config(args)\n    validate_payload(payload, expected_config=config, use_exid=estimator is not None)\n    if payload.get("source_commit", "") != os.environ.get("MOMOK_SOURCE_COMMIT", ""):\n        raise ValueError("checkpoint source commit mismatch")\n    model.load_state_dict(payload["model_state_dict"])\n    optimizer.load_state_dict(payload["optimizer_state_dict"])\n    lr_scheduler.load_state_dict(payload["lr_scheduler_state_dict"])\n    if estimator is not None:\n        estimator.load_state_dict(payload["estimator_state_dict"])\n        optimizer_mi.load_state_dict(payload["optimizer_mi_state_dict"])\n    _move_optimizer_state(optimizer, args.device)\n    _move_optimizer_state(optimizer_mi, args.device)\n    corpus.train_indices = copy.deepcopy(payload["corpus_train_indices"])\n    random.setstate(payload["python_random_state"])\n    np.random.set_state(payload["numpy_rng_state"])\n    torch.set_rng_state(payload["torch_rng_state"])\n    if torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(payload["torch_cuda_rng_state_all"])\n    return payload\n'}
for relative,source in OVERLAY.items():
    target=REPO_DIR/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source)
print('ABLATION_SOURCE_OVERLAY=PASS')


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-q','gdown','tqdm'])
import torch
assert torch.cuda.is_available()
print('GPU=PASS')
print('EFFECTIVE_GPU='+torch.cuda.get_device_name(0))


In [ ]:
archive=Path('/kaggle/working/embeddings.zip')
subprocess.check_call([sys.executable,'-m','gdown','--id','1dKJdJunb11kDtFr5NLfPlFknS7cRdm9W','-O',str(archive)])
extract=Path('/kaggle/working/embeddings_extracted')
extract.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    names=z.namelist()
    for member in (f'embeddings/{DATASET}/img_features.pth',f'embeddings/{DATASET}/text_features.pth'):
        assert names.count(member)==1
        z.extract(member,extract)
dataset_dir=REPO_DIR/'datasets'/DATASET
dataset_dir.mkdir(parents=True,exist_ok=True)
for name in ('img_features.pth','text_features.pth'):
    shutil.copy2(extract/'embeddings'/DATASET/name,dataset_dir/name)
print('SELECTED_IMG_MEMBER=embeddings/'+DATASET+'/img_features.pth')
print('SELECTED_TEXT_MEMBER=embeddings/'+DATASET+'/text_features.pth')


In [ ]:
required=['entity2id.txt','relation2id.txt','train.txt','valid.txt','test.txt','img_features.pth','text_features.pth']
assert not [n for n in required if not (dataset_dir/n).is_file()]
counts={k:sum(1 for _ in (dataset_dir/f).open()) for k,f in {'entities':'entity2id.txt','relations':'relation2id.txt','train':'train.txt','valid':'valid.txt','test':'test.txt'}.items()}
img=torch.load(dataset_dir/'img_features.pth',map_location='cpu',weights_only=False)
txt=torch.load(dataset_dir/'text_features.pth',map_location='cpu',weights_only=False)
assert counts==EXPECTED and list(img.shape)==[EXPECTED['entities'],383] and list(txt.shape)==[EXPECTED['entities'],384]
assert torch.isfinite(img).all() and torch.isfinite(txt).all()
(OUTPUT_DIR/'dataset_manifest.json').write_text(json.dumps({**counts,'image_shape':list(img.shape),'text_shape':list(txt.shape)},indent=2))
print('DATASET_VALIDATION=PASS')


In [ ]:
command=['python', '-u', 'train.py', '--cuda', '0', '--dataset', 'MKG-W', '--variant', 'no_gating_noise', '--lr', '0.001', '--mu', '0.0001', '--dim', '200', '--batch_size', '1024', '--n_exp', '3', '--seed', '10010', '--epochs', '2', '--eval_freq', '1', '--save', '1', '--checkpoint_every', '1', '--checkpoint_dir', '/kaggle/working/momok_ablation/checkpoints', '--output_dir', '/kaggle/working/momok_ablation', '--run_mode', 'smoke', '--max_train_batches', '2']
env=os.environ.copy()
env['PYTHONPATH']=str(REPO_DIR)
log=OUTPUT_DIR/'ablation_smoke.log'
with log.open('w') as handle:
    result=subprocess.run(command,cwd=REPO_DIR,env=env,text=True,stdout=handle,stderr=subprocess.STDOUT)
print('TRAIN_EXIT_CODE='+str(result.returncode))
if result.returncode:
    print('TRAIN_LOG_BEGIN')
    try:
        lines=log.read_text(errors='replace').splitlines()
        for line in lines[-240:]:
            print(line)
    finally:
        print('TRAIN_LOG_END')
    raise RuntimeError('ablation run failed; inspect '+str(log))
print('RUN_MODE_COMPLETE=smoke')


In [ ]:
inventory=sorted(str(p.relative_to(OUTPUT_DIR)) for p in OUTPUT_DIR.rglob('*') if p.is_file())
(OUTPUT_DIR/'output_inventory.json').write_text(json.dumps(inventory,indent=2))
print('OUTPUT_INVENTORY=PASS')
